# V15 — Seed ensemble (dikunci) + **diagnostik plafon representasi**

**Status:** siap jalan, belum pernah dieksekusi siapa pun. Semua angka di bawah adalah angka historis dari `results/`.

---

## 1. Kenapa berhenti menebak arsitektur

Empat eksperimen berturut-turut menaruh struktur lebih kaya di atas fitur frozen 8192-D yang sama. Semuanya rugi:

| Eksperimen | Yang ditambahkan | Delta OOF |
|---|---|---:|
| V10-rich | 17 cabang fixed spatial pooling | −0,00153 (fold 0) |
| V11 | material-query cross-attention | −0,0067 |
| V14 | Gram / bilinear orde-dua | **−0,00142** |
| V13 | (bukan readout) kernel resize | −0,0000115, di bawah noise |

V14 memberi bukti yang paling menutup, dan bukan lewat skornya: **kontrol capacity-matched**-nya (head hidden 640, 5,26 M parameter, tanpa struktur Gram) hanya turun **−0,000214**, sedangkan Gram turun **−0,00142**. Jadi penurunan itu bukan efek "head kebesaran" — struktur Gram sendiri yang merusak. V14 memperbaiki 63 baris dan **merusak 107** (net −44). Ia bukan gagal menemukan sinyal baru; ia menghancurkan sinyal lama.

Menawarkan varian kelima dari kategori yang sama akan jadi tebakan, bukan riset. Jadi notebook ini melakukan dua hal yang **belum pernah** dilakukan proyek ini:

1. **Mengukur plafonnya**, bukan menebak arsitektur berikutnya. Berapa banyak dari 287 error yang secara struktural tidak bisa diperbaiki oleh head apa pun di atas representasi ini?
2. **Menyerang satu-satunya sumber ketidakstabilan yang sudah terukur**: variansi seed.

---

## 2. Variansi seed adalah masalah terbesar yang selama ini diabaikan

Kontrol yang sama, tiga seed (data dari V13, dikonfirmasi ulang V14):

| Seed | OOF Macro F1 | Total error | R/O |
|---:|---:|---:|---:|
| 42 | 0.9901662324 | 287 | 242 |
| 314 | 0.9898990773 | 295 | 249 |
| 2026 | 0.9896446211 | 308 | 267 |

Std **0,000213**, rentang **21 error**. Dua konsekuensi yang belum pernah ditindaklanjuti:

- **Champion 0.9901662324 adalah undian terbaik dari tiga.** Ekspektasi jujur resep V10 ≈ 0,98990 ± 0,00021, bukan 0,99017.
- **Setiap kandidat selama ini diadu melawan undian terbaik itu.** Lihat polanya di V14: penalti Gram −0,00142 di seed 42 tetapi hanya −0,00020 di seed 2026 — paling besar persis di seed tempat kontrol paling beruntung.

Seed ensembling menghapus persis sumber keberuntungan ini. Ia baru saja diizinkan sebagai pengecualian eksplisit terhadap single-model contract, dan ini eksperimen pertama yang memakainya.

**Ini variance reduction, bukan representasi baru — dan notebook ini tidak berpura-pura sebaliknya.** Tapi setelah empat kegagalan representasi, mengurangi variansi yang sudah terukur adalah satu-satunya jalur tersisa dengan mekanisme yang jelas dan biaya compute mendekati nol.

### Apakah ini masih "satu model"?

Backbone tetap satu, beku, dan **tetap satu forward pass per citra** — bagian yang mahal tidak digandakan. Yang dievaluasi K kali hanyalah head 4,2 juta parameter di atas vektor fitur yang sama. Satu backbone, satu arsitektur head, satu aturan agregasi, satu bundle artefak. Tanpa fold ensembling, tanpa model soup, tanpa blending antar-arsitektur.

---

## 3. Yang dikunci sebelum ada satu pun angka

| Item | Nilai terkunci |
|---|---|
| Jumlah seed | **K = 5** |
| Seed | **42, 314, 2026, 7, 1337** |
| Agregasi | **rata-rata seragam probabilitas softmax**, lalu argmax |
| Head | resep champion V10 apa adanya, 4.212.739 parameter |
| Fold | `StratifiedKFold(5, shuffle, random_state=42)`, sha256 `0a06976e…` |
| Noise seed acuan | 0,000213 (terukur V13) |

Seed dipilih sebelum melihat hasil apa pun. Tidak ada seed yang boleh ditambahkan, dibuang, atau ditukar setelah angka muncul — itu akan mengubah eksperimen ini jadi pemilihan post-hoc.

**Perhatikan:** seed di sini hanya mengubah inisialisasi head dan urutan shuffle. **Pembagian fold tetap `random_state=42` untuk semua seed**, supaya seluruhnya tetap protocol-matched terhadap champion.

---

## 4. Gerbang adopsi — sembilan syarat, semuanya wajib

1. Integritas pipeline: head seed-42 tunggal mereproduksi `0.9901662323927614` dalam ±0.0010.
2. Ensemble ≥ **single-seed terbaik + 0,000426** (= 2 × noise seed).
3. Ensemble mengalahkan **setiap** seed tunggal, bukan cuma rata-ratanya.
4. Total error ensemble **turun** vs single-seed terbaik.
5. Error R/O ensemble **turun** vs single-seed terbaik.
6. Ensemble menang ≥ **3 dari 4** fold penilaian vs single-seed terbaik.
7. Electronic F1 tidak turun > 0,0005 vs single-seed terbaik.
8. **Leave-one-seed-out:** kelima sub-ensemble beranggota 4 seed **juga** harus mengalahkan single-seed terbaik. Ini pengganti "stabil lintas seed" yang bermakna untuk sebuah ensemble — membuktikan kemenangannya tidak bergantung pada satu seed beruntung.
9. Tidak ada akses test selama seleksi.

Syarat 2 itu berat dan memang disengaja. Single-seed terbaik dari 5 undian sudah bias ke atas; ensemble harus mengalahkannya **dengan margin**, bukan sekadar menang tipis.

---

## 5. Diagnostik plafon — bagian yang bisa mengubah strategi

Semua berjalan di atas cache fitur, tanpa backbone, dalam hitungan menit. Tidak satu pun menyentuh label: **tidak ada relabel, tidak ada drop, tidak ada koreksi** (kebijakan baru, CONTEXT §16.5).

| Kode | Diagnostik | Pertanyaan yang dijawab |
|---|---|---|
| D1 | Konsensus error lintas seed | Berapa error yang salah di **semua** K seed? Itu batas atas manfaat ensembling. |
| D2 | Konsensus lintas eksperimen | Berapa baris yang salah di **setiap** model yang pernah dibangun (V10/V13/V14/cache teman)? Itu inti yang kebal terhadap perubahan head. |
| D3 | Probe k-NN fold-respecting | Kalau k-NN mentah di ruang 8192-D hampir menyamai MLP, head sudah mendekati batas representasi. Dan untuk tiap error: tetangga terdekatnya berlabel kelas benar atau kelas yang diprediksi? |
| D4 | Proyeksi PCA 2-D | Apakah cluster R dan O benar-benar tumpang tindih, atau ada struktur yang belum dipakai? |
| D5 | Stratifikasi ukuran/aspect per kelas | Apakah error benar-benar terkonsentrasi di resolusi rendah / aspect ekstrem? |
| D6 | **Kemurnian label tetangga** | Untuk tiap baris, berapa fraksi 20 tetangga terdekatnya yang berlabel sama. Baris dengan kemurnian ≈ 0 hampir pasti salah label atau memang ambigu. **Ini estimasi lantai noise label tanpa menyentuh satu label pun.** |

D6 adalah jawaban langsung untuk pertanyaan yang selama ini menggantung: **apakah 0,997 realistis sama sekali.** Kalau jumlah baris berkemurnian-nol sebanding dengan 287, plafonnya adalah datanya, bukan modelnya — dan itu jauh lebih berguna diketahui sekarang daripada setelah tiga eksperimen lagi.

---

## 6. Anggaran runtime

Notebook mencari cache fitur yang kompatibel di `/kaggle/input` dulu. Cache apa pun yang lolos gerbang verifikasi boleh dipakai — bundle V13, V14, atau `timm-embedding-satdat` milik teman satu tim (sudah diaudit lolos di CONTEXT §16.4).

| Skenario | Perkiraan |
|---|---|
| Cache train **dan** test terpasang | **± 25 menit** (backbone tidak pernah dimuat) |
| Cache train saja | ± 45 menit (backbone hanya untuk 1.458 citra test) |
| Tidak ada cache | ± 2 jam (ekstraksi penuh 1,4 jam) |

Verifikasi cache pinjaman itu keras: latih head champion seed 42 lima fold di atasnya dan **wajib** mereproduksi `0.9901662323927614`. Kalau meleset, cache dibuang dan notebook ekstraksi ulang sendiri. Cache teman satu tim lolos audit metadata tetapi `timm_control` mereka menghasilkan 0.9898093823 (297 error) — kemungkinan besar karena resep head mereka berbeda, tapi itu justru alasan gerbang runtime ini ada.

## 7. Sebelum `Run All`

1. **Accelerator `GPU T4 x2`.** Wajib, walaupun backbone mungkin tidak pernah dimuat — diagnostik k-NN memakai GPU dan sharding backbone butuh dua T4 kalau ekstraksi terpicu.
2. **Lampirkan cache fitur kalau ada.** Apa pun dari: bundle V13 (`v13_dinov3_vit7b_model_native_resampling`), bundle V14 (`v14_dinov3_vit7b_second_order_gram_texture`), atau `timm-embedding-satdat`. Notebook menemukan sendiri, memverifikasi, dan memakai yang lolos. Kalau tidak ada, ia ekstraksi sendiri — cuma lebih lama.
3. **Internet ON + Kaggle Secret `HF_TOKEN`** tetap diperlukan sebagai jaring pengaman kalau ekstraksi terpicu. Kalau kedua cache ditemukan, keduanya tidak akan dipakai.
4. **Dataset resmi terpasang**: `train` (9.999 / 3.961 / 12.567), `test` (1.458), `submission.csv` resmi.
5. **Isi `TEAM_NAME`.** V13 dan V14 dua-duanya keluar sebagai `submission_CHANGE_ME.csv`.
6. **Jangan submit hasil notebook ini ke panitia.** Sesuai model kerja siklus eksperimen: kumpulkan kandidat dulu, pilih satu di dekat deadline. Notebook tetap membuat CSV valid sebagai fallback.
7. Sel terakhir mencetak **V15 RESULT BLOCK**. Salin apa adanya.

> **Catatan deadline:** `CONTEXT.md` pernah menulis 31 Juli 16.00 WIB, `PROMPT_V3.md` menulis 30 Juli 16.00 WIB. Sampai ada koreksi eksplisit, rencanakan ke **30 Juli 16.00 WIB**.

In [ ]:
import os

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import gc
import hashlib
import json
import math
import platform
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageFile, ImageOps
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, log_loss
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2
from torchvision.transforms.functional import InterpolationMode
from tqdm.auto import tqdm

# ---------------------------------------------------------------------------
# FILL THIS IN (V13 and V14 both shipped as submission_CHANGE_ME.csv)
# ---------------------------------------------------------------------------
TEAM_NAME = 'CHANGE_ME'

EXPERIMENT_NAME = 'v15_dinov3_vit7b_seed_ensemble_and_ceiling'
REVISION_MARKER = 'v15_seed_ensemble_k5_v1'
EXPERIMENT_PURPOSE = (
    'locked 5-seed probability ensemble of the V10 champion head, plus a representation-ceiling '
    'diagnostic suite that measures how much of the residual error is reachable at all'
)

# ---------------------------------------------------------------------------
# BACKBONE CONTRACT — locked. Not an open question.
# ---------------------------------------------------------------------------
TIMM_MODEL_NAME = 'vit_7b_patch16_dinov3.lvd1689m'
TIMM_HF_REPO_ID = 'timm/vit_7b_patch16_dinov3.lvd1689m'
MODEL_REVISION_REQUEST = 'main'
V10_MODEL_REVISION = 'b40f6c827cb02b13c7a23a76cf598489bb0f94e5'

IMAGE_SIZE = 256
EXPECTED_PATCH_TOKENS = 256
EXPECTED_PREFIX_TOKENS = 5
EXPECTED_REGISTER_TOKENS = 4
EXPECTED_SEQUENCE_TOKENS = 261
EXPECTED_HIDDEN_SIZE = 4096
FIRST_ORDER_BRANCHES = ('final_cls', 'final_patch_mean')
FIRST_ORDER_DIM = len(FIRST_ORDER_BRANCHES) * EXPECTED_HIDDEN_SIZE   # 8192

BACKBONE_DTYPE = torch.float16
BACKBONE_DTYPE_NAME = 'float16'
POOLING_COMPUTE_DTYPE = torch.float32
FEATURE_STORAGE_DTYPE = torch.float16
SHARDING_STRATEGY = 'explicit_contiguous_two_gpu_streamed'
GPU_STATIC_MEMORY_LIMIT_GIB = 10.5
GPU_MIN_RUNTIME_HEADROOM_GIB = 3.0
OUTPUT_SIDE_MODULE_NAMES = ('norm', 'fc_norm', 'head_norm', 'attn_pool', 'head_drop', 'head', 'head_dist')
FEATURE_BATCH_SIZE = 4
FEATURE_LOADER_WORKERS = 2
FEATURE_SHARD_SIZE = 256
FEATURE_MAX_ABS_LIMIT = 1e6

INTERPOLATION_NAME = 'bilinear'   # V13 closed the kernel question; locked to the champion.
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
LARGE_IMAGE_MP_THRESHOLD = 8_000_000
LARGE_IMAGE_MAX_SOURCE_SIDE = 4096
SAFE_DECODE_MAX_SIDE = 1024
PIL_MAX_IMAGE_PIXELS = 200_000_000

# ---------------------------------------------------------------------------
# OFFICIAL DATA CONTRACT
# ---------------------------------------------------------------------------
CLASS_DIR_TO_LABEL = {'0_Recyclable': 0, '1_Electronic': 1, '2_Organic': 2}
CLASS_NAMES = ('Recyclable', 'Electronic', 'Organic')
NUM_CLASSES = len(CLASS_NAMES)
EXPECTED_RAW_COUNTS = {0: 9999, 1: 3961, 2: 12567}
EXPECTED_TRAIN_ROWS = 26527
EXPECTED_TEST_ROWS = 1458
SUBMISSION_COLUMNS = ['id', 'predicted']
ALLOWED_IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# ---------------------------------------------------------------------------
# HEAD PROTOCOL — the V10 champion recipe, unchanged
# ---------------------------------------------------------------------------
FOLD_SEED = 42          # fold split seed, NEVER varies -- keeps every arm protocol-matched
N_FOLDS = 5
DEVELOPMENT_FOLD = 0
ASSESSMENT_FOLDS = (1, 2, 3, 4)
MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 5
HEAD_LEARNING_RATE = 8e-4
HEAD_WEIGHT_DECAY = 1e-4
HEAD_BATCH_SIZE = 256
PREDICT_BATCH_SIZE = 512
GRAD_CLIP_MAX_NORM = 1.0
LABEL_SMOOTHING = 0.05
HEAD_HIDDEN_DIM = 512
HEAD_DROPOUT = 0.40

# ---------------------------------------------------------------------------
# THE LOCKED EXPERIMENTAL VARIABLE
# K and the seed list and the aggregation rule are fixed HERE, before any result
# exists. Adding, dropping or swapping a seed after seeing numbers would turn this
# into post-hoc selection and invalidate the whole experiment.
# ---------------------------------------------------------------------------
ENSEMBLE_SEEDS = (42, 314, 2026, 7, 1337)
ENSEMBLE_K = len(ENSEMBLE_SEEDS)
AGGREGATION_RULE = 'uniform_mean_of_softmax_probabilities_then_argmax'
REFERENCE_SEED = 42     # the champion's seed; used for the pipeline-integrity check

# ---------------------------------------------------------------------------
# LOCKED GATES
# ---------------------------------------------------------------------------
SEED_NOISE_STD = 0.00021296796880071838          # measured in V13, confirmed in V14
ADOPTION_MIN_GAIN_OVER_BEST_SEED = 2.0 * SEED_NOISE_STD   # 0.000426
ADOPTION_MIN_ASSESSMENT_FOLDS_WON = 3
ADOPTION_MAX_ELECTRONIC_F1_DROP = 0.0005
CONTROL_REPRODUCTION_TOLERANCE = 0.0010
RELOAD_TOLERANCE = 1e-5

# Diagnostics (never gates)
KNN_NEIGHBOUR_COUNTS = (1, 5, 20)
PURITY_K = 20
PURITY_LOW_THRESHOLD = 0.10
SIMILARITY_CHUNK_ROWS = 1024

# ---------------------------------------------------------------------------
# MEASURED HISTORICAL FACTS (from results/, never a claim about this run)
# ---------------------------------------------------------------------------
V10_REFERENCE = {
    'fold_assignment_sha256': '0a06976e5dc3cdb0b3f1f5043ff033bf95a40dd2b0cb82154bbf34d701ee305c',
    'labels_sha256': '719d36e9918b1fd09711c06d1b259fca08ca6eb4d973de1388d5e3ddecf22d35',
    'ordered_image_ids_sha256': '210f8f14797be49bad962e81f1da660a257ec28ee2136b54a7516950a66cb2e8',
    'raw_data_sha256': '73bd0d3c17e75e7c0f10e4324ffa9ef81f6224cd2b3ba0e53e9f2aea7fde3212',
    'oof_macro_f1': 0.9901662323927614,
    'oof_total_errors': 287,
    'oof_ro_errors': 242,
    'oof_electronic_errors': 16,
    'oof_f1_electronic': 0.994328922495274,
    'oof_high_confidence_errors': 33,
    'per_fold_macro_f1': {'0': 0.9897706675032462, '1': 0.9903469894805695, '2': 0.9908291600511604,
                          '3': 0.9890345290445208, '4': 0.9908498966736979},
    'head_parameter_count': 4212739,
    'full_training_epochs': 5,
    'test_predicted_class_distribution': {'0': 520, '1': 201, '2': 737},
}
PRIOR_SEED_RESULTS = {
    '42': {'macro_f1': 0.9901662323927614, 'total_errors': 287, 'ro_errors': 242},
    '314': {'macro_f1': 0.9898990773387165, 'total_errors': 295, 'ro_errors': 249},
    '2026': {'macro_f1': 0.9896446210609852, 'total_errors': 308, 'ro_errors': 267},
}
REJECTED_HISTORY = {
    'v10_rich_17_branch_spatial': 'fold0 -0.00153',
    'v11_material_query_attention': '-0.0067',
    'v13_bicubic_resize_kernel': '-0.0000115, feature cosine vs bilinear 0.99941',
    'v14_gram_second_order': '-0.0014216; capacity-matched control only -0.000214, so the Gram structure '
                             'itself was the harm, not head capacity',
    'v14_grouped_validation_leakage_cost': '-0.000186 only, so duplicate leakage is near-negligible',
}

# ---------------------------------------------------------------------------
# CONTRACT ASSERTIONS
# ---------------------------------------------------------------------------
assert IMAGE_SIZE == 256 and INTERPOLATION_NAME == 'bilinear'
assert EXPECTED_SEQUENCE_TOKENS == EXPECTED_PREFIX_TOKENS + EXPECTED_PATCH_TOKENS
assert EXPECTED_PREFIX_TOKENS == EXPECTED_REGISTER_TOKENS + 1
assert FOLD_SEED == 42 and N_FOLDS == 5 and ASSESSMENT_FOLDS == (1, 2, 3, 4)
assert sum(EXPECTED_RAW_COUNTS.values()) == EXPECTED_TRAIN_ROWS
assert ENSEMBLE_K == 5 and len(set(ENSEMBLE_SEEDS)) == ENSEMBLE_K, 'K and the seed list are locked'
assert REFERENCE_SEED in ENSEMBLE_SEEDS, 'the champion seed must be inside the ensemble'
assert ENSEMBLE_SEEDS[0] == REFERENCE_SEED, 'run the reference seed first so integrity fails fast'
assert AGGREGATION_RULE == 'uniform_mean_of_softmax_probabilities_then_argmax'
assert abs(ADOPTION_MIN_GAIN_OVER_BEST_SEED - 0.00042593593760143676) < 1e-12

TEST_ACCESS_STATE = {'test_images_accessed': False, 'manual_test_labels_accessed': False}
NOTEBOOK_STARTED_AT = time.perf_counter()

print('Experiment      :', EXPERIMENT_NAME)
print('Locked seeds    :', ENSEMBLE_SEEDS, '(K = {})'.format(ENSEMBLE_K))
print('Aggregation     :', AGGREGATION_RULE)
print('Fold split seed :', FOLD_SEED, '(never varies -- only head init/shuffle uses the ensemble seeds)')
print('Adoption needs  : ensemble >= best single seed + {:.6f}  (= 2 x measured seed noise {:.6f})'.format(
    ADOPTION_MIN_GAIN_OVER_BEST_SEED, SEED_NOISE_STD))
print('Head            : V10 champion recipe, {:,} parameters, unchanged'.format(
    V10_REFERENCE['head_parameter_count']))
print()
print('No relabelling, no dropping, no manual label correction anywhere in this notebook.')
print('Train and validation both use the official raw labels (CONTEXT 16.5).')
if TEAM_NAME == 'CHANGE_ME':
    print()
    print('!' * 78)
    print('!! TEAM_NAME is still "CHANGE_ME" -- V13 and V14 both shipped that way.         !!')
    print('!' * 78)

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(FOLD_SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
torch.set_float32_matmul_precision('high')
Image.MAX_IMAGE_PIXELS = PIL_MAX_IMAGE_PIXELS
ImageFile.LOAD_TRUNCATED_IMAGES = False

KAGGLE_WORKING = Path('/kaggle/working')
if not KAGGLE_WORKING.is_dir():
    raise RuntimeError('This notebook requires a Kaggle runtime with /kaggle/working and GPU T4 x2.')
KAGGLE_INPUT = Path('/kaggle/input')
ARTIFACT_DIR = KAGGLE_WORKING / EXPERIMENT_NAME
CACHE_DIR = ARTIFACT_DIR / 'feature_cache'
HEADS_DIR = ARTIFACT_DIR / 'heads'
FIGURES_DIR = ARTIFACT_DIR / 'figures'
for directory in (ARTIFACT_DIR, CACHE_DIR, HEADS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)
SELECTION_DECISION_PATH = ARTIFACT_DIR / 'selection_decision.json'


def json_default(value):
    if isinstance(value, (np.integer, np.floating, np.bool_)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError('Not JSON serializable: {}'.format(type(value).__name__))


def sha256_json(value):
    payload = json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False, default=json_default)
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()


def write_json_atomic(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        json.dump(value, handle, indent=2, sort_keys=True, default=json_default, allow_nan=False)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


def write_csv_atomic(frame, path, index=False):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=index)
    os.replace(temporary, path)


def write_npy_atomic(array, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp.npy')
    np.save(temporary, array)
    os.replace(temporary, path)


def atomic_torch_save(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    if temporary.exists():
        temporary.unlink()
    with temporary.open('wb') as handle:
        torch.save(value, handle)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


def read_json(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return json.load(handle)


def gpu_memory_rows():
    return [{'device_id': i, 'allocated_gib': torch.cuda.memory_allocated(i) / 2 ** 30,
             'reserved_gib': torch.cuda.memory_reserved(i) / 2 ** 30}
            for i in range(torch.cuda.device_count())]


def cuda_memory_snapshot(label):
    rows = []
    for device_id in range(torch.cuda.device_count()):
        free_bytes, total_bytes = torch.cuda.mem_get_info(device_id)
        rows.append({'label': label, 'device_id': device_id, 'free_gib': free_bytes / 2 ** 30,
                     'total_gib': total_bytes / 2 ** 30,
                     'allocated_gib': torch.cuda.memory_allocated(device_id) / 2 ** 30})
    return rows


def release_cuda_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def sanitize_for_filename(value):
    cleaned = ''.join(c if c.isalnum() or c in {'-', '_'} else '_' for c in str(value))
    return cleaned.strip('_') or 'UNSET'


def elapsed_minutes():
    return (time.perf_counter() - NOTEBOOK_STARTED_AT) / 60.0


import accelerate
import safetensors
import timm
import timm.data

ENVIRONMENT_INFO = {
    'python_version': sys.version, 'platform': platform.platform(),
    'torch_version': torch.__version__, 'cuda_version': torch.version.cuda,
    'timm_version': timm.__version__, 'accelerate_version': accelerate.__version__,
    'safetensors_version': safetensors.__version__,
    'numpy_version': np.__version__, 'pandas_version': pd.__version__,
}

if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError('V15 requires exactly two visible CUDA devices; set the accelerator to "GPU T4 x2".')
GPU_RUNTIME_INFO = []
for device_id in (0, 1):
    properties = torch.cuda.get_device_properties(device_id)
    if 'TESLA T4' not in properties.name.upper() and properties.name.upper().strip() != 'T4':
        raise RuntimeError('cuda:{} is "{}", not a Tesla T4.'.format(device_id, properties.name))
    GPU_RUNTIME_INFO.append({'device_id': device_id, 'name': properties.name,
                             'total_vram_gib': properties.total_memory / 2 ** 30})

if not KAGGLE_INPUT.is_dir():
    raise RuntimeError('Attach the competition dataset: /kaggle/input was not found.')
blocked_label_files = [str(p) for p in KAGGLE_INPUT.rglob('*.csv')
                       if 'manual' in p.name.lower().replace('_', '-') and 'label' in p.name.lower()]
if blocked_label_files:
    raise RuntimeError('Competition safety guard: manual test-label CSVs attached:'
                       + '\n' + '\n'.join(blocked_label_files))

LOCKED_CONTRACT = {
    'schema_version': 1, 'revision_marker': REVISION_MARKER, 'experiment_name': EXPERIMENT_NAME,
    'experiment_purpose': EXPERIMENT_PURPOSE,
    'backbone': {'backend': 'timm', 'model': TIMM_MODEL_NAME, 'frozen': True,
                 'dtype': BACKBONE_DTYPE_NAME, 'sharding': SHARDING_STRATEGY,
                 'unchanged_from_v10': True},
    'preprocessing': {'interpolation': INTERPOLATION_NAME, 'image_size': IMAGE_SIZE,
                      'geometry': 'direct_square, antialias True, no crop, no letterbox, no augmentation',
                      'mean': list(IMAGENET_MEAN), 'std': list(IMAGENET_STD),
                      'unchanged_from_v10': True},
    'readout': {'branches': list(FIRST_ORDER_BRANCHES), 'dim': FIRST_ORDER_DIM,
                'definition': 'final normalised CLS concatenated with the mean of the 256 true patch tokens',
                'register_tokens_excluded': True, 'unchanged_from_v10': True},
    'head': {'architecture': 'LayerNorm(8192) -> Linear(512) -> GELU -> Dropout(0.4) -> Linear(3)',
             'parameter_count_expected': V10_REFERENCE['head_parameter_count'], 'unchanged_from_v10': True},
    'the_only_change': {
        'name': 'seed_ensemble',
        'k': ENSEMBLE_K, 'seeds': list(ENSEMBLE_SEEDS), 'aggregation': AGGREGATION_RULE,
        'locked_before_any_result': True,
        'rationale': ('measured seed noise std 0.000213 with a 21-error range means the champion 0.9901662324 '
                      'is the luckiest of three draws; probability averaging over a locked seed set removes '
                      'exactly that lottery'),
    },
    'single_model_contract': {
        'fold_ensembling': False, 'model_soup': False, 'cross_architecture_blending': False,
        'test_time_augmentation': False, 'multi_crop_inference': False,
        'horizontal_flip_inference': False, 'multiple_backbone_families': False,
        'manual_prediction_correction': False,
        'seed_ensembling': True,
        'seed_ensembling_note': ('explicitly permitted by the user as an exception. One backbone, one frozen '
                                 'forward pass per image, one head architecture, one locked aggregation rule. '
                                 'Only the 4.2M-parameter head is evaluated K times on the same feature vector.'),
    },
    'label_policy': {
        'relabelling': False, 'dropping': False, 'manual_correction': False,
        'blind_human_review': False,
        'note': 'train and validation both use the official raw labels, without exception (CONTEXT 16.5)',
    },
    'gates': {
        'control_reproduction_tolerance': CONTROL_REPRODUCTION_TOLERANCE,
        'min_gain_over_best_single_seed': ADOPTION_MIN_GAIN_OVER_BEST_SEED,
        'must_beat_every_single_seed': True,
        'min_assessment_folds_won': ADOPTION_MIN_ASSESSMENT_FOLDS_WON,
        'max_electronic_f1_drop': ADOPTION_MAX_ELECTRONIC_F1_DROP,
        'leave_one_seed_out_must_all_win': True,
        'measured_seed_noise_std': SEED_NOISE_STD,
    },
    'historical_reference_v10': V10_REFERENCE,
    'prior_seed_results': PRIOR_SEED_RESULTS,
    'rejected_history': REJECTED_HISTORY,
}
LOCKED_CONTRACT_SIGNATURE = sha256_json(LOCKED_CONTRACT)
write_json_atomic({'environment': ENVIRONMENT_INFO, 'gpus': GPU_RUNTIME_INFO},
                  ARTIFACT_DIR / 'environment.json')
write_json_atomic({'locked_contract': LOCKED_CONTRACT,
                   'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE},
                  ARTIFACT_DIR / 'locked_contract.json')

print(json.dumps(ENVIRONMENT_INFO, indent=2))
for info in GPU_RUNTIME_INFO:
    print('cuda:{device_id}: {name}, {total_vram_gib:.2f} GiB'.format(**info))
print('Manual test-label guard passed.')
print('Locked contract signature:', LOCKED_CONTRACT_SIGNATURE)

In [ ]:
def discover_named_dirs(base, wanted, max_depth=4, prune_names=()):
    if not base.is_dir():
        return []
    found, stack = [], [(base, 0)]
    while stack:
        current, depth = stack.pop()
        try:
            children = list(current.iterdir())
        except OSError:
            continue
        for child in children:
            if not child.is_dir():
                continue
            lowered = child.name.lower()
            if lowered in prune_names:
                continue          # pruned without enumeration: test stays untouched during selection
            if lowered == wanted:
                found.append(child)
            elif depth < max_depth:
                stack.append((child, depth + 1))
    return found


def exact_raw_train_signature(train_dir):
    if not train_dir.is_dir():
        return None
    try:
        root_entries = list(train_dir.iterdir())
        if {p.name for p in root_entries} != set(CLASS_DIR_TO_LABEL):
            return None
        if any(p.is_symlink() or not p.is_dir() for p in root_entries):
            return None
        counts = {}
        for class_dir_name, label in CLASS_DIR_TO_LABEL.items():
            entries = list((train_dir / class_dir_name).iterdir())
            if any(e.is_symlink() or not e.is_file() or e.suffix.lower() not in ALLOWED_IMAGE_SUFFIXES
                   for e in entries):
                return None
            counts[label] = len(entries)
        return counts if counts == EXPECTED_RAW_COUNTS else None
    except OSError:
        return None


def resolve_train_dir():
    candidates = [Path.cwd() / 'data' / 'train', KAGGLE_WORKING / 'data' / 'train']
    candidates.extend(discover_named_dirs(KAGGLE_INPUT, 'train', prune_names={'test'}))
    matches = {c.resolve() for c in candidates if exact_raw_train_signature(c) == EXPECTED_RAW_COUNTS}
    if len(matches) != 1:
        rendered = '\n'.join('  - {}'.format(p) for p in sorted(matches, key=str)) or '  (none)'
        raise RuntimeError('Expected exactly one official raw train directory; found {}:'.format(len(matches))
                           + '\n' + rendered)
    return matches.pop()


def load_rgb_image(path):
    """Identical decode path to V7 / V10 / V13 / V14, including the decoder-memory guard."""
    try:
        with Image.open(path) as image:
            source_width, source_height = image.size
            if source_width <= 0 or source_height <= 0:
                raise ValueError('invalid image dimensions')
            needs_downscale = (source_width * source_height >= LARGE_IMAGE_MP_THRESHOLD
                               or max(source_width, source_height) > LARGE_IMAGE_MAX_SOURCE_SIDE)
            image.draft('RGB', (SAFE_DECODE_MAX_SIDE, SAFE_DECODE_MAX_SIDE))
            rgb_image = ImageOps.exif_transpose(image).convert('RGB')
            if needs_downscale:
                rgb_image.thumbnail((SAFE_DECODE_MAX_SIDE, SAFE_DECODE_MAX_SIDE), Image.Resampling.LANCZOS)
            return rgb_image.copy()
    except Exception as exc:
        raise RuntimeError('Image cannot be decoded: {}'.format(path)) from exc


IMAGE_TRANSFORM = v2.Compose([
    v2.ToImage(),
    v2.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=InterpolationMode.BILINEAR, antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

PREPROCESS_CONFIG = {
    'open': 'PIL, EXIF transpose, RGB conversion',
    'large_image_guard': {'megapixel_threshold': LARGE_IMAGE_MP_THRESHOLD,
                          'max_source_side': LARGE_IMAGE_MAX_SOURCE_SIDE,
                          'downscaled_to_max_side': SAFE_DECODE_MAX_SIDE, 'resample': 'LANCZOS',
                          'purpose': 'decoder memory safety only, never a predictive feature'},
    'resize_geometry': 'direct_square', 'spatial_override': {'height': IMAGE_SIZE, 'width': IMAGE_SIZE},
    'interpolation': INTERPOLATION_NAME, 'antialias': True, 'rescale_factor': 1.0 / 255.0,
    'image_mean': list(IMAGENET_MEAN), 'image_std': list(IMAGENET_STD),
    'center_crop': False, 'letterbox': False, 'augmentation': 'none', 'feature_view': 'orig',
}
PREPROCESS_SIGNATURE = sha256_json(PREPROCESS_CONFIG)


class ImageRows(Dataset):
    """Deterministic, order-preserving image reader for frozen feature extraction."""

    def __init__(self, paths):
        self.paths = [str(p) for p in paths]

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, position):
        return IMAGE_TRANSFORM(load_rgb_image(self.paths[position]))


def read_source_size(path):
    """Header-only read. Used for error stratification only, never as a model feature."""
    try:
        with Image.open(path) as image:
            width, height = image.size
            return int(width), int(height)
    except Exception:
        return 0, 0


def index_raw_train(train_dir):
    rows, observed_counts = [], {}
    for class_dir_name, label in CLASS_DIR_TO_LABEL.items():
        class_dir = train_dir / class_dir_name
        entries = sorted(class_dir.iterdir(), key=lambda p: p.name)
        invalid = [e.name for e in entries
                   if e.is_symlink() or not e.is_file() or e.suffix.lower() not in ALLOWED_IMAGE_SUFFIXES]
        if invalid:
            raise RuntimeError('Unsupported entries in {}: {}'.format(class_dir, invalid[:10]))
        observed_counts[label] = len(entries)
        for path in entries:
            rows.append({'image_id': path.relative_to(train_dir).as_posix(), 'label': label,
                         'class_name': CLASS_NAMES[label], 'path': str(path.resolve()),
                         'file_size_bytes': path.stat().st_size})
    if observed_counts != EXPECTED_RAW_COUNTS:
        raise RuntimeError('Raw class counts mismatch: {} vs {}.'.format(observed_counts, EXPECTED_RAW_COUNTS))
    frame = pd.DataFrame(rows).sort_values('image_id').reset_index(drop=True)
    if len(frame) != EXPECTED_TRAIN_ROWS or not frame['image_id'].is_unique:
        raise RuntimeError('Raw-train row count or image_id uniqueness failed.')
    return frame


TRAIN_DIR = resolve_train_dir()
train_df = index_raw_train(TRAIN_DIR)

# Fold reconstruction: byte-for-byte the V7 / V10 / V13 / V14 definition. FOLD_SEED never varies.
train_df['fold'] = -1
splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=FOLD_SEED)
for fold_index, (_, valid_positions) in enumerate(splitter.split(train_df, train_df['label'])):
    train_df.loc[valid_positions, 'fold'] = fold_index
train_df['fold'] = train_df['fold'].astype(np.int64)
if set(train_df['fold'].tolist()) != set(range(N_FOLDS)):
    raise RuntimeError('Five-fold assignment is incomplete.')

ORDERED_IMAGE_IDS_SHA256 = sha256_json(train_df['image_id'].tolist())
LABELS_SHA256 = sha256_json(train_df['label'].astype(int).tolist())
FOLD_ASSIGNMENT_SHA256 = sha256_json(train_df[['image_id', 'label', 'fold']].to_dict('records'))
RAW_DATA_SHA256 = sha256_json(train_df[['image_id', 'label', 'file_size_bytes']].to_dict('records'))

PROTOCOL_MATCH_CHECKS = {
    'ordered_image_ids_match': ORDERED_IMAGE_IDS_SHA256 == V10_REFERENCE['ordered_image_ids_sha256'],
    'labels_match': LABELS_SHA256 == V10_REFERENCE['labels_sha256'],
    'fold_assignment_matches': FOLD_ASSIGNMENT_SHA256 == V10_REFERENCE['fold_assignment_sha256'],
    'raw_data_matches': RAW_DATA_SHA256 == V10_REFERENCE['raw_data_sha256'],
}
failed_protocol = [k for k, v in PROTOCOL_MATCH_CHECKS.items() if not v]
if failed_protocol:
    raise RuntimeError('Protocol match against V10 failed: {}. Expected fold sha256 {}, got {}.'.format(
        failed_protocol, V10_REFERENCE['fold_assignment_sha256'], FOLD_ASSIGNMENT_SHA256))

# Source-size census for diagnostic D5. Analysis stratum only; explicitly not a model input.
print('Reading image headers for the resolution census (analysis only, never a feature)...')
sizes = [read_source_size(p) for p in tqdm(train_df['path'].tolist(), desc='image headers')]
train_df['source_width'] = [s[0] for s in sizes]
train_df['source_height'] = [s[1] for s in sizes]
train_df['source_area'] = train_df['source_width'] * train_df['source_height']
train_df['is_upsampled'] = train_df['source_area'] < (IMAGE_SIZE * IMAGE_SIZE)
train_df['aspect_ratio'] = np.where(
    train_df['source_height'] > 0,
    np.maximum(train_df['source_width'], train_df['source_height'])
    / np.maximum(np.minimum(train_df['source_width'], train_df['source_height']), 1),
    1.0)

DATA_CONTRACT = {
    'schema_version': 1, 'revision_marker': REVISION_MARKER,
    'canonical_data': 'official raw labeled train images only',
    'raw_train_root': str(TRAIN_DIR), 'row_count': int(len(train_df)),
    'ordered_image_ids_sha256': ORDERED_IMAGE_IDS_SHA256, 'labels_sha256': LABELS_SHA256,
    'raw_data_sha256': RAW_DATA_SHA256, 'fold_assignment_sha256': FOLD_ASSIGNMENT_SHA256,
    'fold_definition': {'implementation': 'sklearn StratifiedKFold', 'n_splits': N_FOLDS,
                        'shuffle': True, 'random_state': FOLD_SEED,
                        'ordering': 'relative official image_id ascending before the split'},
    'preprocess_config': PREPROCESS_CONFIG, 'preprocess_signature': PREPROCESS_SIGNATURE,
    'external_labeled_images_used': False, 'metadata_used_as_feature': False,
    'dropped_or_relabeled_rows': 0,
    'label_policy': 'official raw labels for both training and validation, no exceptions',
    'source_dimensions_recorded_for': 'error stratification only',
    'upsampled_row_count': int(train_df['is_upsampled'].sum()),
    'test_images_accessed': TEST_ACCESS_STATE['test_images_accessed'],
}
DATA_CONTRACT_SIGNATURE = sha256_json(DATA_CONTRACT)
write_json_atomic({'data_contract': DATA_CONTRACT, 'data_contract_signature': DATA_CONTRACT_SIGNATURE},
                  ARTIFACT_DIR / 'data_contract.json')
write_csv_atomic(train_df[['image_id', 'label', 'class_name', 'fold', 'source_width', 'source_height',
                           'is_upsampled', 'aspect_ratio']], ARTIFACT_DIR / 'fold_assignments.csv')

cached_labels = torch.tensor(train_df['label'].to_numpy(), dtype=torch.long)
cached_folds = torch.tensor(train_df['fold'].to_numpy(), dtype=torch.long)
FOLD_VALUES = cached_folds.numpy()
LABEL_VALUES = cached_labels.numpy()

print()
print('Official raw train root :', TRAIN_DIR)
print('Rows                    :', len(train_df))
print('Fold assignment sha256  :', FOLD_ASSIGNMENT_SHA256, '-> matches V10')
print('Raw data sha256         :', RAW_DATA_SHA256, '-> matches V10')
print('Upsampled to 256x256    : {} of {} ({:.1f}%)'.format(
    int(train_df['is_upsampled'].sum()), len(train_df),
    100.0 * train_df['is_upsampled'].mean()))
print('Data contract signature :', DATA_CONTRACT_SIGNATURE)

In [ ]:
HEAD_DEVICE = 'cuda:0'


class CachedRows(Dataset):
    def __init__(self, features, indices, labels=None):
        self.features = features
        self.indices = [int(i) for i in indices]
        self.labels = labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        index = self.indices[position]
        label = int(self.labels[index]) if self.labels is not None else -1
        return self.features[index], label, index


class ChampionHead(nn.Module):
    """The adopted V10 readout, unchanged. Only its init/shuffle seed varies across the ensemble."""

    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.LayerNorm(FIRST_ORDER_DIM),
            nn.Linear(FIRST_ORDER_DIM, HEAD_HIDDEN_DIM),
            nn.GELU(),
            nn.Dropout(HEAD_DROPOUT),
            nn.Linear(HEAD_HIDDEN_DIM, NUM_CLASSES),
        )

    def forward(self, features):
        return self.network(features.flatten(1))


def softmax_numpy(logits):
    shifted = logits - logits.max(axis=1, keepdims=True)
    exponentials = np.exp(shifted)
    probabilities = exponentials / exponentials.sum(axis=1, keepdims=True)
    if not np.isfinite(probabilities).all():
        raise RuntimeError('Head probabilities contain NaN or Inf.')
    return probabilities


def predict_probabilities(model, features, indices):
    index_list = [int(i) for i in indices]
    loader = DataLoader(CachedRows(features, index_list), batch_size=PREDICT_BATCH_SIZE,
                        shuffle=False, num_workers=0, pin_memory=True)
    parts, observed = [], []
    model.eval()
    with torch.inference_mode():
        for batch_features, _labels, batch_indices in loader:
            logits = model(batch_features.to(device=HEAD_DEVICE, dtype=torch.float32, non_blocking=True))
            if not torch.isfinite(logits).all().item():
                raise RuntimeError('Head logits contain NaN or Inf.')
            parts.append(logits.float().cpu())
            observed.extend(int(i) for i in batch_indices.tolist())
    if observed != index_list:
        raise RuntimeError('Prediction order changed inside the DataLoader.')
    stacked = torch.cat(parts).numpy().astype(np.float64)
    if stacked.shape != (len(index_list), NUM_CLASSES):
        raise RuntimeError('Head prediction matrix has shape {}.'.format(stacked.shape))
    return softmax_numpy(stacked)


def distribution_summary(values):
    values = np.asarray(values, dtype=np.float64)
    if not len(values):
        return {'count': 0, 'mean': None, 'std': None, 'q10': None, 'median': None, 'q90': None}
    return {'count': int(len(values)), 'mean': float(values.mean()), 'std': float(values.std()),
            'q10': float(np.quantile(values, 0.10)), 'median': float(np.quantile(values, 0.50)),
            'q90': float(np.quantile(values, 0.90))}


def metric_bundle(labels, probabilities):
    labels = np.asarray(labels, dtype=np.int64)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    if probabilities.shape != (len(labels), NUM_CLASSES) or not np.isfinite(probabilities).all():
        raise RuntimeError('Metric input probabilities are incomplete or non-finite.')
    predictions = probabilities.argmax(axis=1)
    matrix = confusion_matrix(labels, predictions, labels=list(range(NUM_CLASSES)))
    report = classification_report(labels, predictions, labels=list(range(NUM_CLASSES)),
                                   target_names=list(CLASS_NAMES), output_dict=True, zero_division=0)
    top_two = np.sort(probabilities, axis=1)[:, -2:]
    confidence, margin = top_two[:, 1], top_two[:, 1] - top_two[:, 0]
    correct = predictions == labels
    r_to_o, o_to_r = int(matrix[0, 2]), int(matrix[2, 0])
    return {
        'rows': int(len(labels)),
        'macro_f1': float(f1_score(labels, predictions, labels=list(range(NUM_CLASSES)), average='macro')),
        'accuracy': float(accuracy_score(labels, predictions)),
        'log_loss': float(log_loss(labels, probabilities, labels=list(range(NUM_CLASSES)))),
        'f1_recyclable': float(report['Recyclable']['f1-score']),
        'f1_electronic': float(report['Electronic']['f1-score']),
        'f1_organic': float(report['Organic']['f1-score']),
        'confusion_matrix': matrix.tolist(),
        'total_errors': int((~correct).sum()),
        'recyclable_to_organic': r_to_o, 'organic_to_recyclable': o_to_r, 'ro_errors': r_to_o + o_to_r,
        'electronic_errors': int(matrix[1].sum() - matrix[1, 1]),
        'high_confidence_errors': int(((~correct) & (confidence >= 0.90)).sum()),
        'confidence_summary': {'all': distribution_summary(confidence),
                               'correct': distribution_summary(confidence[correct]),
                               'errors': distribution_summary(confidence[~correct])},
        'margin_summary': {'all': distribution_summary(margin),
                           'errors': distribution_summary(margin[~correct])},
    }


def class_weights_from(label_values):
    counts = np.bincount(label_values, minlength=NUM_CLASSES)
    if (counts <= 0).any():
        raise RuntimeError('A training partition is missing a class: {}.'.format(counts.tolist()))
    return counts, torch.tensor(counts.sum() / (NUM_CLASSES * counts),
                                dtype=torch.float32, device=HEAD_DEVICE)


def run_training_epochs(model, features, train_indices, epochs, seed,
                        criterion, optimizer, scheduler, on_epoch_end):
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(CachedRows(features, train_indices, cached_labels), batch_size=HEAD_BATCH_SIZE,
                        shuffle=True, generator=generator, num_workers=0, pin_memory=True, drop_last=False)
    for epoch in range(1, epochs + 1):
        started = time.perf_counter()
        model.train()
        loss_sum, seen = 0.0, 0
        learning_rate = optimizer.param_groups[0]['lr']
        for batch_features, batch_labels, _indices in loader:
            batch_features = batch_features.to(device=HEAD_DEVICE, dtype=torch.float32, non_blocking=True)
            batch_labels = batch_labels.to(device=HEAD_DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(batch_features), batch_labels)
            if not torch.isfinite(loss).item():
                raise RuntimeError('Non-finite loss at epoch {}.'.format(epoch))
            loss.backward()
            gradient_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_MAX_NORM)
            if not torch.isfinite(gradient_norm).item():
                raise RuntimeError('Non-finite head gradient norm at epoch {}.'.format(epoch))
            optimizer.step()
            loss_sum += loss.item() * len(batch_labels)
            seen += len(batch_labels)
        scheduler.step()
        if on_epoch_end(epoch, loss_sum / seen, learning_rate, time.perf_counter() - started):
            break
    del loader


def train_fold_head(features, validation_fold, seed, persist=False):
    """One fold, one seed, the locked champion protocol. Always trained from scratch."""
    train_indices = np.flatnonzero(FOLD_VALUES != validation_fold)
    valid_indices = np.flatnonzero(FOLD_VALUES == validation_fold)
    valid_labels = LABEL_VALUES[valid_indices]
    class_counts, class_weights = class_weights_from(LABEL_VALUES[train_indices])

    seed_everything(seed)
    model = ChampionHead().to(HEAD_DEVICE)
    parameter_count = sum(p.numel() for p in model.parameters())
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=HEAD_LEARNING_RATE, weight_decay=HEAD_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

    state = {'best_macro_f1': -1.0, 'best_log_loss': float('inf'), 'best_epoch': None,
             'best_state': None, 'stale': 0}
    started = time.perf_counter()

    def on_epoch_end(epoch, train_loss, learning_rate, epoch_seconds):
        probabilities = predict_probabilities(model, features, valid_indices)
        predictions = probabilities.argmax(axis=1)
        macro = float(f1_score(valid_labels, predictions, labels=list(range(NUM_CLASSES)), average='macro'))
        loss_value = float(log_loss(valid_labels, probabilities, labels=list(range(NUM_CLASSES))))
        improved = macro > state['best_macro_f1'] + 1e-8 or (
            abs(macro - state['best_macro_f1']) <= 1e-8 and loss_value < state['best_log_loss'])
        if improved:
            state.update({'best_macro_f1': macro, 'best_log_loss': loss_value, 'best_epoch': epoch,
                          'stale': 0,
                          'best_state': {n: t.detach().cpu().clone() for n, t in model.state_dict().items()}})
        else:
            state['stale'] += 1
        return state['stale'] >= EARLY_STOPPING_PATIENCE

    run_training_epochs(model, features, train_indices, MAX_EPOCHS, seed,
                        criterion, optimizer, scheduler, on_epoch_end)
    if state['best_state'] is None:
        raise RuntimeError('Head training produced no valid checkpoint.')
    model.load_state_dict(state['best_state'], strict=True)
    probabilities = predict_probabilities(model, features, valid_indices)
    metrics = metric_bundle(valid_labels, probabilities)
    metrics.update({'seed': seed, 'validation_fold': validation_fold, 'best_epoch': state['best_epoch'],
                    'hit_epoch_cap': bool(state['best_epoch'] == MAX_EPOCHS),
                    'parameter_count': parameter_count,
                    'training_seconds': time.perf_counter() - started,
                    'train_class_counts': class_counts.tolist()})
    if persist:
        write_json_atomic(metrics, HEADS_DIR / 'fold{}_seed{}_metrics.json'.format(validation_fold, seed))
    del model, criterion, optimizer, scheduler
    state['best_state'] = None
    release_cuda_cache()
    return metrics, valid_indices, probabilities


def run_five_fold_oof(features, seed, persist=False, quiet=False):
    """Full five-fold OOF. Every row is predicted exactly once -- no fold ensembling."""
    per_fold = []
    oof_probabilities = np.zeros((len(train_df), NUM_CLASSES), dtype=np.float64)
    covered = np.zeros(len(train_df), dtype=bool)
    for fold in range(N_FOLDS):
        metrics, valid_indices, probabilities = train_fold_head(features, fold, seed, persist=persist)
        oof_probabilities[valid_indices] = probabilities
        covered[valid_indices] = True
        per_fold.append(metrics)
        if not quiet:
            print('    seed={:<5} fold={} macro_f1={:.10f} err={} R/O={} elecF1={:.6f} epoch={}{}'.format(
                seed, fold, metrics['macro_f1'], metrics['total_errors'], metrics['ro_errors'],
                metrics['f1_electronic'], metrics['best_epoch'],
                ' CAP' if metrics['hit_epoch_cap'] else ''))
    if not covered.all():
        raise RuntimeError('OOF coverage is incomplete.')
    oof = metric_bundle(LABEL_VALUES, oof_probabilities)
    oof.update({'seed': seed,
                'per_fold_macro_f1': {str(m['validation_fold']): m['macro_f1'] for m in per_fold},
                'best_epochs': {str(m['validation_fold']): m['best_epoch'] for m in per_fold},
                'folds_hitting_epoch_cap': int(sum(1 for m in per_fold if m['hit_epoch_cap']))})
    return per_fold, oof, oof_probabilities


probe_head = ChampionHead()
HEAD_PARAMETER_COUNT = sum(p.numel() for p in probe_head.parameters())
del probe_head
if HEAD_PARAMETER_COUNT != V10_REFERENCE['head_parameter_count']:
    raise RuntimeError('Head parameter count {:,} != the adopted V10 head ({:,}).'.format(
        HEAD_PARAMETER_COUNT, V10_REFERENCE['head_parameter_count']))
print('Champion head ready: {:,} parameters (matches V10 exactly).'.format(HEAD_PARAMETER_COUNT))

In [ ]:
# ---------------------------------------------------------------------------
# BACKBONE LOADER -- DEFINITIONS ONLY. Nothing here runs unless a cache is missing.
# This is the V13/V14 loader verbatim; it has run end to end on Kaggle twice and
# reproduced the champion bit-exactly both times, so it is not re-derived.
# ---------------------------------------------------------------------------
from accelerate import dispatch_model, init_empty_weights
from accelerate.utils import set_module_tensor_to_device
from safetensors import safe_open

DISPATCH_FAILURE_HINT = (
    'The DINOv3 ViT-7B backbone could not be placed across the two Tesla T4 GPUs.' + '\n'
    '  1. Confirm the accelerator is "GPU T4 x2" and no other process holds GPU memory.' + '\n'
    '  2. Restart the session so both GPUs start empty, then rerun from the top.' + '\n'
    'CPU offload, disk offload, single-GPU placement and smaller backbones are all forbidden.'
)
BACKBONE_STATE = {'model': None, 'blocks': None, 'input_device': None, 'prefix_tokens': None}


def _tensor_bytes(tensor):
    element = torch.empty(0, dtype=BACKBONE_DTYPE).element_size() if tensor.is_floating_point() \
        else tensor.element_size()
    return tensor.numel() * element


def _module_bytes(module):
    total = sum(_tensor_bytes(p) for _, p in module.named_parameters(recurse=True))
    for _, submodule in module.named_modules():
        for name, buffer in submodule._buffers.items():
            if buffer is not None and name not in submodule._non_persistent_buffers_set:
                total += _tensor_bytes(buffer)
    return total


def _plan_two_gpu_placement(model):
    blocks = getattr(model, 'blocks', None)
    if blocks is None or len(blocks) < 2:
        raise RuntimeError('The timm model does not expose an ordered "blocks" list.')
    num_blocks = len(blocks)
    root_names, root_bytes = [], 0
    for name, parameter in model._parameters.items():
        if parameter is not None:
            root_names.append(name)
            root_bytes += _tensor_bytes(parameter)
    for name, buffer in model._buffers.items():
        if buffer is not None and name not in model._non_persistent_buffers_set:
            root_names.append(name)
            root_bytes += _tensor_bytes(buffer)
    block_bytes = [_module_bytes(b) for b in blocks]
    input_side, output_side = [], []
    for child_name, child in model.named_children():
        if child_name == 'blocks':
            continue
        (output_side if child_name in OUTPUT_SIDE_MODULE_NAMES else input_side).append(
            (child_name, _module_bytes(child)))
    input_bytes = root_bytes + sum(s for _, s in input_side)
    output_bytes = sum(s for _, s in output_side)
    candidates = []
    for split in range(1, num_blocks):
        gpu0 = input_bytes + sum(block_bytes[:split])
        gpu1 = output_bytes + sum(block_bytes[split:])
        candidates.append((max(gpu0, gpu1), gpu0, split, gpu1))
    candidates.sort()
    _, gpu0_bytes, split_index, gpu1_bytes = candidates[0]

    device_map = {name: 'cuda:0' for name in root_names}
    for child_name, _ in input_side:
        device_map[child_name] = 'cuda:0'
    for index in range(num_blocks):
        device_map['blocks.{}'.format(index)] = 'cuda:0' if index < split_index else 'cuda:1'
    for child_name, _ in output_side:
        device_map[child_name] = 'cuda:1'
    if '' in device_map or set(device_map.values()) != {'cuda:0', 'cuda:1'}:
        raise RuntimeError('The planned device map is invalid.')
    return {'strategy': SHARDING_STRATEGY, 'transformer_block_count': num_blocks,
            'contiguous_split_index': split_index,
            'estimated_static_bytes': {'cuda:0': int(gpu0_bytes), 'cuda:1': int(gpu1_bytes)},
            'estimated_static_gib': {'cuda:0': gpu0_bytes / 2 ** 30, 'cuda:1': gpu1_bytes / 2 ** 30},
            'device_map': device_map}


def _enforce_budget(plan):
    limit = GPU_STATIC_MEMORY_LIMIT_GIB * 2 ** 30
    for device_id in (0, 1):
        key = 'cuda:{}'.format(device_id)
        planned = plan['estimated_static_bytes'][key]
        free_bytes, total_bytes = torch.cuda.mem_get_info(device_id)
        if planned > limit or (total_bytes - planned) / 2 ** 30 < GPU_MIN_RUNTIME_HEADROOM_GIB \
                or free_bytes < planned:
            raise RuntimeError('Placement budget rejected on {}.'.format(key) + '\n' + DISPATCH_FAILURE_HINT)
    return plan


def _device_for(tensor_name, device_map):
    best_name, best_device = None, None
    for module_name, device in device_map.items():
        if tensor_name == module_name or tensor_name.startswith(module_name + '.'):
            if best_name is None or len(module_name) > len(best_name):
                best_name, best_device = module_name, device
    if best_device is None:
        raise RuntimeError('No device assignment covers checkpoint tensor {!r}.'.format(tensor_name))
    return best_device


def build_backbone():
    """Download, stream, dispatch and audit. Called lazily, only when a cache is missing."""
    from huggingface_hub import model_info, snapshot_download

    token = os.environ.get('HF_TOKEN')
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient

            token = UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:
            token = None
    if not token:
        raise RuntimeError(
            'A feature cache was not found, so the backbone must be loaded, but no HF_TOKEN is available.'
            + '\n' + 'Either attach a compatible feature cache, or add a Kaggle Secret named HF_TOKEN whose '
            'account has accepted the DINOv3 license.')

    revision = model_info(TIMM_HF_REPO_ID, revision=MODEL_REVISION_REQUEST, token=token).sha
    if not revision or len(revision) != 40:
        raise RuntimeError('The Hub did not return an immutable 40-character commit SHA.')
    snapshot_dir = snapshot_download(repo_id=TIMM_HF_REPO_ID, revision=revision, token=token,
                                     allow_patterns=['*.safetensors', '*.safetensors.index.json', '*.json'])
    del token

    with init_empty_weights(include_buffers=False):
        model = timm.create_model(TIMM_MODEL_NAME, pretrained=False, num_classes=0, img_size=IMAGE_SIZE)
    snapshot = Path(snapshot_dir)
    index_path = snapshot / 'model.safetensors.index.json'
    if index_path.is_file():
        names = sorted(set(read_json(index_path).get('weight_map', {}).values()))
    else:
        available = sorted(p.name for p in snapshot.glob('*.safetensors'))
        names = ['model.safetensors'] if 'model.safetensors' in available else available
    checkpoint_files = [snapshot / n for n in names]
    if not checkpoint_files or any(not p.is_file() for p in checkpoint_files):
        raise RuntimeError('Checkpoint files are missing in {}.'.format(snapshot))

    plan = _enforce_budget(_plan_two_gpu_placement(model))
    print('Planned contiguous split at block {} of {}.'.format(
        plan['contiguous_split_index'], plan['transformer_block_count']))
    model_tensor_names = set(model.state_dict().keys())
    checkpoint_names = set()
    for checkpoint_file in checkpoint_files:
        with safe_open(str(checkpoint_file), framework='pt', device='cpu') as handle:
            keys = sorted(handle.keys())
            checkpoint_names.update(keys)
            for key in tqdm(keys, desc='streaming {}'.format(checkpoint_file.name)):
                if key not in model_tensor_names:
                    continue
                tensor = handle.get_tensor(key)
                if tensor.is_floating_point():
                    tensor = tensor.to(BACKBONE_DTYPE)
                set_module_tensor_to_device(model, key, _device_for(key, plan['device_map']),
                                            value=tensor, dtype=BACKBONE_DTYPE)
                del tensor
    missing = sorted(model_tensor_names - checkpoint_names)
    if missing:
        raise RuntimeError('Checkpoint missing {} model tensors, e.g. {}.'.format(len(missing), missing[:10]))
    if [n for n, t in list(model.named_parameters()) + list(model.named_buffers())
            if t.device.type == 'meta']:
        raise RuntimeError('Some tensors were never materialised.')
    model = dispatch_model(model, device_map=plan['device_map'], force_hooks=True)
    model.eval().requires_grad_(False)

    if sum(p.numel() for p in model.parameters() if p.requires_grad) != 0:
        raise RuntimeError('The pretrained backbone is not completely frozen.')
    if {str(p.device) for p in model.parameters()} != {'cuda:0', 'cuda:1'}:
        raise RuntimeError('Parameters are not split across exactly both GPUs.')
    if {p.dtype for p in model.parameters() if p.is_floating_point()} != {BACKBONE_DTYPE}:
        raise RuntimeError('Floating parameters are not uniformly FP16.')
    block_devices = []
    for block in model.blocks:
        devices = {str(p.device) for p in block.parameters()}
        if len(devices) != 1:
            raise RuntimeError('A transformer block is split across devices.')
        block_devices.append(devices.pop())
    transitions = [i for i in range(1, len(block_devices)) if block_devices[i] != block_devices[i - 1]]
    if len(transitions) != 1 or block_devices[0] != 'cuda:0' or block_devices[-1] != 'cuda:1':
        raise RuntimeError('Blocks must form exactly two contiguous device ranges, cuda:0 then cuda:1.')
    if not [n for n, m in model.named_modules() if hasattr(m, '_hf_hook')]:
        raise RuntimeError('No Accelerate hooks are attached.')
    forbidden = {''.join(('Data', 'Parallel')), ''.join(('Distributed', 'Data', 'Parallel'))}
    if type(model).__name__ in forbidden:
        raise RuntimeError('Replication wrapper detected; only model sharding is allowed.')

    prefix_tokens = int(getattr(model, 'num_prefix_tokens'))
    if prefix_tokens != EXPECTED_PREFIX_TOKENS or int(model.num_features) != EXPECTED_HIDDEN_SIZE:
        raise RuntimeError('Runtime token contract does not match the locked contract.')
    data_config = timm.data.resolve_model_data_config(model)
    if not np.allclose(tuple(float(v) for v in data_config['mean']), IMAGENET_MEAN, atol=1e-6):
        raise RuntimeError('timm normalization does not match the locked ImageNet constants.')

    BACKBONE_STATE.update({'model': model, 'blocks': list(model.blocks),
                           'input_device': next(model.patch_embed.proj.parameters()).device,
                           'prefix_tokens': prefix_tokens, 'revision': revision,
                           'block_devices': block_devices})
    write_json_atomic({'repository': TIMM_HF_REPO_ID, 'immutable_revision': revision,
                       'revision_matches_v10': revision == V10_MODEL_REVISION,
                       'timm_model_name': TIMM_MODEL_NAME, 'frozen': True,
                       'block_devices': block_devices, 'placement_strategy': SHARDING_STRATEGY,
                       'automatic_device_map_string_used': False, 'cpu_or_disk_offload': False,
                       'data_parallel_used': False,
                       'backbone_documented_for_final_report': (
                           'DINOv3 ViT-7B/16, timm repo timm/vit_7b_patch16_dinov3.lvd1689m, LVD-1689M '
                           'checkpoint, revision {}, 100 percent frozen in FP16, sharded across two Tesla '
                           'T4 GPUs. Only a 4.2M-parameter classifier head was trained, on the official '
                           'training images. No external labeled images and no test labels were '
                           'used.'.format(revision)),
                       'environment': ENVIRONMENT_INFO, 'gpus': GPU_RUNTIME_INFO},
                      ARTIFACT_DIR / 'model_manifest.json')
    print('Backbone dispatched: cuda:0 x{} then cuda:1 x{}; revision {} (matches V10: {}).'.format(
        block_devices.count('cuda:0'), block_devices.count('cuda:1'),
        revision, revision == V10_MODEL_REVISION))
    return model


def extract_champion_features(pixel_values):
    """[B, 3, 256, 256] FP32 -> [B, 2, 4096] FP16 on CPU. Exactly the V10 champion readout."""
    model = BACKBONE_STATE['model']
    if model is None:
        raise RuntimeError('The backbone has not been built.')
    with torch.inference_mode():
        model_input = pixel_values.to(device=BACKBONE_STATE['input_device'],
                                      dtype=BACKBONE_DTYPE, non_blocking=True)
        tokens = model.forward_features(model_input)
        if tokens.ndim != 3 or tokens.shape[1] != EXPECTED_SEQUENCE_TOKENS \
                or tokens.shape[2] != EXPECTED_HIDDEN_SIZE:
            raise RuntimeError('forward_features returned shape {}.'.format(tuple(tokens.shape)))
        prefix = BACKBONE_STATE['prefix_tokens']
        cls_fp32 = tokens[:, 0, :].to(POOLING_COMPUTE_DTYPE)
        patches = tokens[:, prefix:, :].to(POOLING_COMPUTE_DTYPE)
        if patches.shape[1] != EXPECTED_PATCH_TOKENS:
            raise RuntimeError('True-patch token count is {}.'.format(patches.shape[1]))
        pooled = torch.stack((cls_fp32, patches.mean(dim=1)), dim=1)
        if not torch.isfinite(pooled).all().item():
            raise RuntimeError('Pooled features contain NaN or Inf.')
        if float(pooled.abs().max().item()) > FEATURE_MAX_ABS_LIMIT:
            raise RuntimeError('Pooled feature range is implausible.')
        return pooled.detach().to(device='cpu', dtype=FEATURE_STORAGE_DTYPE)


def extract_split(frame, split_name):
    """Extract a whole split, sharded and written atomically so a broken session can resume."""
    if BACKBONE_STATE['model'] is None:
        build_backbone()
    shard_dir = CACHE_DIR / split_name
    shard_dir.mkdir(parents=True, exist_ok=True)
    features = torch.empty((len(frame), len(FIRST_ORDER_BRANCHES), EXPECTED_HIDDEN_SIZE),
                           dtype=FEATURE_STORAGE_DTYPE)
    boundaries = [(s, min(s + FEATURE_SHARD_SIZE, len(frame)))
                  for s in range(0, len(frame), FEATURE_SHARD_SIZE)]
    written = 0
    started = time.perf_counter()
    for start, end in tqdm(boundaries, desc='{} extraction'.format(split_name)):
        shard_path = shard_dir / '{}_{:06d}_{:06d}.pt'.format(split_name, start, end)
        window_ids = frame.iloc[start:end]['image_id'].tolist()
        if shard_path.is_file():
            try:
                payload = torch.load(shard_path, map_location='cpu', weights_only=False)
                if payload.get('image_ids') == window_ids and torch.is_tensor(payload.get('features')) \
                        and payload['features'].shape[0] == end - start:
                    features[start:end] = payload['features']
                    del payload
                    continue
            except Exception:
                pass
        loader = DataLoader(ImageRows(frame.iloc[start:end]['path'].tolist()),
                            batch_size=FEATURE_BATCH_SIZE, shuffle=False,
                            num_workers=FEATURE_LOADER_WORKERS, pin_memory=True, persistent_workers=False)
        cursor = start
        for batch in loader:
            pooled = extract_champion_features(batch)
            features[cursor:cursor + pooled.shape[0]] = pooled
            cursor += pooled.shape[0]
            del pooled
        if cursor != end:
            raise RuntimeError('Shard extraction produced the wrong row count.')
        atomic_torch_save({'features': features[start:end].clone(), 'image_ids': window_ids}, shard_path)
        written += end - start
        del loader
    seconds = time.perf_counter() - started
    print('{} extraction done: {} rows written, {:.1f} min ({:.2f} img/s).'.format(
        split_name, written, seconds / 60.0, (written / seconds) if seconds > 0 and written else 0.0))
    return features


print('Backbone loader defined. It will run only if a compatible feature cache is not found.')

In [ ]:
# ---------------------------------------------------------------------------
# FEATURE ACQUISITION
# Prefer a mounted cache, but only after it earns it. Any cache -- V13, V14, or the
# teammate's timm-embedding-satdat bundle -- is accepted purely on the evidence that
# training the champion head on it reproduces 0.9901662323927614. Metadata alone is
# not enough: the teammate bundle passes every metadata check yet their own control
# scored 0.9898093823, so only a runtime reproduction gate can settle it.
# ---------------------------------------------------------------------------
FIRST_ORDER_TAIL = (len(FIRST_ORDER_BRANCHES), EXPECTED_HIDDEN_SIZE)


def _find_feature_tensor(payload, rows):
    """Schema-agnostic: accept any tensor whose non-row dimensions flatten to 8192."""
    if torch.is_tensor(payload):
        candidates = [payload]
    elif isinstance(payload, dict):
        candidates = [value for value in payload.values() if torch.is_tensor(value)]
    else:
        return None
    for tensor in candidates:
        if tensor.ndim < 2 or int(tensor.shape[0]) != rows:
            continue
        flat = 1
        for dimension in tensor.shape[1:]:
            flat *= int(dimension)
        if flat == FIRST_ORDER_DIM:
            return tensor.reshape(rows, *FIRST_ORDER_TAIL)
    return None


def discover_cache_dirs(split_name):
    directories = set()
    if KAGGLE_INPUT.is_dir():
        for path in KAGGLE_INPUT.rglob('{}_000000_*.pt'.format(split_name)):
            directories.add(path.parent)
    local = CACHE_DIR / split_name
    if local.is_dir() and any(local.glob('{}_000000_*.pt'.format(split_name))):
        directories.add(local)
    return sorted(directories, key=str)


def load_cache_dir(directory, split_name, frame):
    """Assemble a [N, 2, 4096] FP16 tensor from a shard directory, or return None."""
    expected_rows = len(frame)
    shards = sorted(directory.glob('{}_*.pt'.format(split_name)), key=lambda p: p.name)
    if not shards:
        return None
    features = torch.empty((expected_rows,) + FIRST_ORDER_TAIL, dtype=FEATURE_STORAGE_DTYPE)
    covered = np.zeros(expected_rows, dtype=bool)
    for shard_path in shards:
        stem_parts = shard_path.stem.split('_')
        if len(stem_parts) < 3:
            return None
        try:
            start, end = int(stem_parts[-2]), int(stem_parts[-1])
        except ValueError:
            return None
        if start < 0 or end > expected_rows or end <= start:
            return None
        try:
            payload = torch.load(shard_path, map_location='cpu', weights_only=False)
        except Exception:
            return None
        tensor = _find_feature_tensor(payload, end - start)
        if tensor is None:
            del payload
            return None
        if isinstance(payload, dict) and isinstance(payload.get('image_ids'), list):
            if payload['image_ids'] != frame.iloc[start:end]['image_id'].tolist():
                del payload
                return None
        features[start:end] = tensor.to(FEATURE_STORAGE_DTYPE)
        covered[start:end] = True
        del payload, tensor
    if not covered.all():
        return None
    if not torch.isfinite(features.to(POOLING_COMPUTE_DTYPE)).all().item():
        return None
    return features


def verify_champion_reproduction(features):
    """The only test that matters: does the champion head land on the champion number?"""
    _, oof, _ = run_five_fold_oof(features, REFERENCE_SEED, persist=False, quiet=True)
    delta = oof['macro_f1'] - V10_REFERENCE['oof_macro_f1']
    return {
        'macro_f1': oof['macro_f1'], 'delta': delta,
        'total_errors': oof['total_errors'], 'ro_errors': oof['ro_errors'],
        'within_tolerance': bool(abs(delta) <= CONTROL_REPRODUCTION_TOLERANCE),
        'exactly_reproduced': bool(delta == 0.0),
    }, oof


CACHE_ACQUISITION = {'train': None, 'test': None}
print('=' * 78)
print('FEATURE ACQUISITION -- train   [t+{:.0f} min]'.format(elapsed_minutes()))
print('=' * 78)

TRAIN_FEATURES = None
train_candidates = discover_cache_dirs('train')
print('Candidate train cache directories found: {}'.format(len(train_candidates)))
for candidate in train_candidates:
    print('  - {}'.format(candidate))

for candidate in train_candidates[:4]:
    print()
    print('Trying {} ...'.format(candidate))
    assembled = load_cache_dir(candidate, 'train', train_df)
    if assembled is None:
        print('  rejected: shards do not assemble into a complete [{}, 2, 4096] tensor.'.format(
            len(train_df)))
        continue
    print('  assembled {} rows; verifying by reproducing the champion (this takes ~2 min)...'.format(
        assembled.shape[0]))
    verification, verification_oof = verify_champion_reproduction(assembled)
    print('    champion head seed {} on this cache -> macro_f1 {:.10f} (delta {:+.10f}), '
          'errors {}, R/O {}'.format(REFERENCE_SEED, verification['macro_f1'], verification['delta'],
                                     verification['total_errors'], verification['ro_errors']))
    if verification['within_tolerance']:
        TRAIN_FEATURES = assembled
        CACHE_ACQUISITION['train'] = {'source': 'mounted_cache', 'directory': str(candidate),
                                      'verification': verification}
        print('  ACCEPTED. Extraction skipped, saving about 1.4 h of GPU time.')
        break
    print('  rejected: it does not reproduce the champion within {:.4f}.'.format(
        CONTROL_REPRODUCTION_TOLERANCE))
    del assembled

if TRAIN_FEATURES is None:
    print()
    print('No mounted train cache earned acceptance. Extracting from the frozen backbone instead.')
    TRAIN_FEATURES = extract_split(train_df, 'train')
    CACHE_ACQUISITION['train'] = {'source': 'fresh_extraction', 'directory': str(CACHE_DIR / 'train')}
    release_cuda_cache()

if tuple(TRAIN_FEATURES.shape) != (len(train_df),) + FIRST_ORDER_TAIL:
    raise RuntimeError('Train feature tensor has shape {}.'.format(tuple(TRAIN_FEATURES.shape)))
write_json_atomic(CACHE_ACQUISITION, ARTIFACT_DIR / 'cache_acquisition.json')
print()
print('Train features ready: {} rows, {:.2f} GiB, source = {}'.format(
    TRAIN_FEATURES.shape[0],
    TRAIN_FEATURES.numel() * TRAIN_FEATURES.element_size() / 2 ** 30,
    CACHE_ACQUISITION['train']['source']))
print('Elapsed: {:.0f} min'.format(elapsed_minutes()))

In [ ]:
print('=' * 78)
print('FIVE-FOLD OOF FOR EACH LOCKED SEED   [t+{:.0f} min]'.format(elapsed_minutes()))
print('=' * 78)

SEED_PER_FOLD, SEED_OOF, SEED_PROBABILITIES = {}, {}, {}
for seed in ENSEMBLE_SEEDS:
    print()
    print('--- seed {} ---'.format(seed))
    per_fold, oof, probabilities = run_five_fold_oof(TRAIN_FEATURES, seed, persist=True)
    SEED_PER_FOLD[seed] = per_fold
    SEED_OOF[seed] = oof
    SEED_PROBABILITIES[seed] = probabilities
    print('  OOF macro_f1={:.10f} errors={} R/O={} elecF1={:.10f} logloss={:.10f} capped={}/5'.format(
        oof['macro_f1'], oof['total_errors'], oof['ro_errors'], oof['f1_electronic'],
        oof['log_loss'], oof['folds_hitting_epoch_cap']))
    write_npy_atomic(probabilities.astype(np.float32),
                     ARTIFACT_DIR / 'oof_probabilities_seed{}.npy'.format(seed))
    write_json_atomic(oof, ARTIFACT_DIR / 'oof_metrics_seed{}.json'.format(seed))

# ---------------------------------------------------------------------------
# PIPELINE INTEGRITY: the reference seed must reproduce the champion.
# ---------------------------------------------------------------------------
reference_oof = SEED_OOF[REFERENCE_SEED]
reference_delta = reference_oof['macro_f1'] - V10_REFERENCE['oof_macro_f1']
CONTROL_REPRODUCTION = {
    'reference_seed': REFERENCE_SEED,
    'v10_recorded_macro_f1': V10_REFERENCE['oof_macro_f1'],
    'v15_reference_seed_macro_f1': reference_oof['macro_f1'],
    'macro_f1_delta': reference_delta,
    'tolerance': CONTROL_REPRODUCTION_TOLERANCE,
    'within_tolerance': bool(abs(reference_delta) <= CONTROL_REPRODUCTION_TOLERANCE),
    'exactly_reproduced': bool(reference_delta == 0.0),
    'total_error_delta': reference_oof['total_errors'] - V10_REFERENCE['oof_total_errors'],
    'ro_error_delta': reference_oof['ro_errors'] - V10_REFERENCE['oof_ro_errors'],
    'feature_source': CACHE_ACQUISITION['train']['source'],
    'v13_v14_precedent': 'both reproduced the champion with macro_f1_delta exactly 0.0',
}
PIPELINE_INTEGRITY_OK = CONTROL_REPRODUCTION['within_tolerance']
write_json_atomic(CONTROL_REPRODUCTION, ARTIFACT_DIR / 'control_reproduction.json')

print()
print('PIPELINE INTEGRITY: seed {} -> {:.10f} vs champion {:.10f}, delta {:+.10f} -> {}'.format(
    REFERENCE_SEED, reference_oof['macro_f1'], V10_REFERENCE['oof_macro_f1'], reference_delta,
    'PASS' if PIPELINE_INTEGRITY_OK else 'FAIL'))

# ---------------------------------------------------------------------------
# THE LOCKED ENSEMBLE
# ---------------------------------------------------------------------------
def ensemble_probabilities(seeds):
    stack = np.stack([SEED_PROBABILITIES[s] for s in seeds], axis=0)
    averaged = stack.mean(axis=0)
    row_sums = averaged.sum(axis=1, keepdims=True)
    if not np.isfinite(averaged).all() or (row_sums <= 0).any():
        raise RuntimeError('Ensemble probabilities are degenerate.')
    return averaged / row_sums


ENSEMBLE_OOF_PROBABILITIES = ensemble_probabilities(ENSEMBLE_SEEDS)
ENSEMBLE_OOF = metric_bundle(LABEL_VALUES, ENSEMBLE_OOF_PROBABILITIES)
write_npy_atomic(ENSEMBLE_OOF_PROBABILITIES.astype(np.float32),
                 ARTIFACT_DIR / 'oof_probabilities_ensemble.npy')

# Per-fold ensemble metrics, so the fold-win gate compares like with like.
ENSEMBLE_PER_FOLD = {}
for fold in range(N_FOLDS):
    mask = FOLD_VALUES == fold
    ENSEMBLE_PER_FOLD[fold] = metric_bundle(LABEL_VALUES[mask], ENSEMBLE_OOF_PROBABILITIES[mask])

single_macro = {s: SEED_OOF[s]['macro_f1'] for s in ENSEMBLE_SEEDS}
BEST_SINGLE_SEED = max(single_macro, key=single_macro.get)
BEST_SINGLE_OOF = SEED_OOF[BEST_SINGLE_SEED]
MEAN_SINGLE_MACRO = float(np.mean(list(single_macro.values())))
OBSERVED_SEED_STD = float(np.std(list(single_macro.values())))

GAIN_OVER_BEST_SEED = ENSEMBLE_OOF['macro_f1'] - BEST_SINGLE_OOF['macro_f1']
GAIN_OVER_MEAN_SEED = ENSEMBLE_OOF['macro_f1'] - MEAN_SINGLE_MACRO
BEATS_EVERY_SEED = all(ENSEMBLE_OOF['macro_f1'] > single_macro[s] for s in ENSEMBLE_SEEDS)
ELECTRONIC_F1_DROP = BEST_SINGLE_OOF['f1_electronic'] - ENSEMBLE_OOF['f1_electronic']

best_single_per_fold = {int(m['validation_fold']): m for m in SEED_PER_FOLD[BEST_SINGLE_SEED]}
ASSESSMENT_FOLD_GAINS = {
    str(fold): ENSEMBLE_PER_FOLD[fold]['macro_f1'] - best_single_per_fold[fold]['macro_f1']
    for fold in ASSESSMENT_FOLDS
}
FOLDS_WON = sum(1 for g in ASSESSMENT_FOLD_GAINS.values() if g > 0)
DEVELOPMENT_FOLD_GAIN = (ENSEMBLE_PER_FOLD[DEVELOPMENT_FOLD]['macro_f1']
                         - best_single_per_fold[DEVELOPMENT_FOLD]['macro_f1'])

# Leave-one-seed-out: every K-1 sub-ensemble must also beat the best single seed.
LOSO_RESULTS = {}
for dropped in ENSEMBLE_SEEDS:
    subset = tuple(s for s in ENSEMBLE_SEEDS if s != dropped)
    subset_metrics = metric_bundle(LABEL_VALUES, ensemble_probabilities(subset))
    LOSO_RESULTS[str(dropped)] = {
        'seeds_used': list(subset), 'macro_f1': subset_metrics['macro_f1'],
        'total_errors': subset_metrics['total_errors'], 'ro_errors': subset_metrics['ro_errors'],
        'beats_best_single_seed': bool(subset_metrics['macro_f1'] > BEST_SINGLE_OOF['macro_f1']),
    }
LOSO_ALL_WIN = all(v['beats_best_single_seed'] for v in LOSO_RESULTS.values())

ADOPTION_GATE_CHECKS = {
    'pipeline_integrity_reference_seed_reproduces_v10': bool(PIPELINE_INTEGRITY_OK),
    'gain_over_best_single_seed_at_least_two_sigma': bool(
        GAIN_OVER_BEST_SEED >= ADOPTION_MIN_GAIN_OVER_BEST_SEED),
    'beats_every_single_seed': bool(BEATS_EVERY_SEED),
    'total_errors_decrease_vs_best_seed': bool(
        ENSEMBLE_OOF['total_errors'] < BEST_SINGLE_OOF['total_errors']),
    'ro_errors_decrease_vs_best_seed': bool(ENSEMBLE_OOF['ro_errors'] < BEST_SINGLE_OOF['ro_errors']),
    'wins_at_least_three_of_four_assessment_folds': bool(
        FOLDS_WON >= ADOPTION_MIN_ASSESSMENT_FOLDS_WON),
    'electronic_f1_stable': bool(ELECTRONIC_F1_DROP <= ADOPTION_MAX_ELECTRONIC_F1_DROP),
    'leave_one_seed_out_all_beat_best_single_seed': bool(LOSO_ALL_WIN),
    'no_test_access_during_selection': bool(not TEST_ACCESS_STATE['test_images_accessed']),
}
ENSEMBLE_ADOPTED = all(ADOPTION_GATE_CHECKS.values())

seed_table = pd.DataFrame([
    {'arm': 'seed {}'.format(s), 'macro_f1': SEED_OOF[s]['macro_f1'],
     'total_errors': SEED_OOF[s]['total_errors'], 'ro_errors': SEED_OOF[s]['ro_errors'],
     'f1_electronic': SEED_OOF[s]['f1_electronic'], 'log_loss': SEED_OOF[s]['log_loss'],
     'folds_at_epoch_cap': SEED_OOF[s]['folds_hitting_epoch_cap']}
    for s in ENSEMBLE_SEEDS
] + [
    {'arm': 'ENSEMBLE k={}'.format(ENSEMBLE_K), 'macro_f1': ENSEMBLE_OOF['macro_f1'],
     'total_errors': ENSEMBLE_OOF['total_errors'], 'ro_errors': ENSEMBLE_OOF['ro_errors'],
     'f1_electronic': ENSEMBLE_OOF['f1_electronic'], 'log_loss': ENSEMBLE_OOF['log_loss'],
     'folds_at_epoch_cap': None}
])
write_csv_atomic(seed_table, ARTIFACT_DIR / 'seed_and_ensemble_table.csv')

print()
print('=' * 78)
print('SEED AND ENSEMBLE RESULTS')
print('=' * 78)
display(seed_table)
print()
print('Best single seed        : {} at {:.10f}'.format(BEST_SINGLE_SEED, BEST_SINGLE_OOF['macro_f1']))
print('Mean single-seed        : {:.10f}   (observed std {:.8f}, V13 measured {:.8f})'.format(
    MEAN_SINGLE_MACRO, OBSERVED_SEED_STD, SEED_NOISE_STD))
print('Ensemble                : {:.10f}'.format(ENSEMBLE_OOF['macro_f1']))
print('  gain over best seed   : {:+.10f}   (gate needs >= {:+.6f})'.format(
    GAIN_OVER_BEST_SEED, ADOPTION_MIN_GAIN_OVER_BEST_SEED))
print('  gain over mean seed   : {:+.10f}'.format(GAIN_OVER_MEAN_SEED))
print('  errors {} -> {} | R/O {} -> {}'.format(
    BEST_SINGLE_OOF['total_errors'], ENSEMBLE_OOF['total_errors'],
    BEST_SINGLE_OOF['ro_errors'], ENSEMBLE_OOF['ro_errors']))
print('  fold 0 gain           : {:+.10f}'.format(DEVELOPMENT_FOLD_GAIN))
print('  assessment fold gains : {}'.format({k: round(v, 8) for k, v in ASSESSMENT_FOLD_GAINS.items()}))
print('  folds won             : {} of 4'.format(FOLDS_WON))
print()
print('Leave-one-seed-out (each drops one seed, K-1 = {} remain):'.format(ENSEMBLE_K - 1))
for dropped, row in LOSO_RESULTS.items():
    print('  drop {:<5} -> {:.10f} errors {} {}'.format(
        dropped, row['macro_f1'], row['total_errors'],
        'beats best single' if row['beats_best_single_seed'] else 'DOES NOT beat best single'))
print()
for name, passed in ADOPTION_GATE_CHECKS.items():
    print('  [{}] {}'.format('PASS' if passed else 'FAIL', name))
print()
print('DECISION: {}'.format('ADOPT the seed ensemble' if ENSEMBLE_ADOPTED else 'REJECT the seed ensemble'))
print('Elapsed: {:.0f} min'.format(elapsed_minutes()))

In [ ]:
# ---------------------------------------------------------------------------
# CEILING DIAGNOSTICS -- never gates, and they never touch a label.
# No relabelling, no dropping, no correction. Every count below is descriptive only.
# Wrapped so a diagnostic failure can never destroy a run that already paid for features.
# ---------------------------------------------------------------------------
DIAGNOSTICS = {'attempted': True, 'succeeded': False, 'is_a_gate': False}
try:
    print('=' * 78)
    print('CEILING DIAGNOSTICS   [t+{:.0f} min]'.format(elapsed_minutes()))
    print('=' * 78)

    reference_predictions = SEED_PROBABILITIES[REFERENCE_SEED].argmax(axis=1)
    ensemble_predictions = ENSEMBLE_OOF_PROBABILITIES.argmax(axis=1)

    # ---- D1: how many rows are wrong under how many of the K seeds -------------
    wrong_stack = np.stack([SEED_PROBABILITIES[s].argmax(axis=1) != LABEL_VALUES
                            for s in ENSEMBLE_SEEDS], axis=0)
    wrong_count = wrong_stack.sum(axis=0)
    seed_consensus = {str(k): int((wrong_count == k).sum()) for k in range(ENSEMBLE_K + 1)}
    always_wrong = int((wrong_count == ENSEMBLE_K).sum())
    sometimes_wrong = int(((wrong_count > 0) & (wrong_count < ENSEMBLE_K)).sum())
    print()
    print('D1 seed consensus over K={} seeds:'.format(ENSEMBLE_K))
    for k in range(ENSEMBLE_K + 1):
        print('    wrong in {} of {} seeds : {}'.format(k, ENSEMBLE_K, seed_consensus[str(k)]))
    print('    -> {} rows are wrong under EVERY seed. Ensembling can never fix those.'.format(always_wrong))
    print('    -> {} rows flip between seeds. That is the entire headroom for seed ensembling.'.format(
        sometimes_wrong))

    # ---- D2: consensus across every experiment whose OOF predictions we can find --
    cross_experiment = {}
    error_sets = {'v15_seed{}'.format(REFERENCE_SEED): set(np.flatnonzero(
        reference_predictions != LABEL_VALUES).tolist())}
    if KAGGLE_INPUT.is_dir():
        for csv_path in sorted(KAGGLE_INPUT.rglob('oof_predictions_*.csv')):
            try:
                frame = pd.read_csv(csv_path, usecols=['image_id', 'label', 'prediction'])
            except Exception:
                continue
            if len(frame) != len(train_df):
                continue
            wrong_ids = set(frame.loc[frame['prediction'] != frame['label'], 'image_id'].tolist())
            key = '{}::{}'.format(csv_path.parent.name, csv_path.stem)
            error_sets[key] = wrong_ids
            cross_experiment[key] = len(wrong_ids)
    id_by_index = train_df['image_id'].to_numpy()
    normalised_sets = []
    for key, values in error_sets.items():
        if values and isinstance(next(iter(values)), (int, np.integer)):
            normalised_sets.append(set(id_by_index[list(values)]))
        else:
            normalised_sets.append(set(values))
    universal_errors = set.intersection(*normalised_sets) if len(normalised_sets) > 1 else set()
    print()
    print('D2 cross-experiment consensus over {} prediction sets:'.format(len(normalised_sets)))
    for key, count in sorted(cross_experiment.items()):
        print('    {:<58} {} errors'.format(key[:58], count))
    if len(normalised_sets) > 1:
        print('    -> {} rows are wrong in EVERY model found. That is the head-invariant core.'.format(
            len(universal_errors)))
    else:
        print('    -> only this run was found; mount prior result bundles to widen this comparison.')

    # ---- shared neighbour machinery (fold-respecting, so nothing leaks) ---------
    max_k = max(max(KNN_NEIGHBOUR_COUNTS), PURITY_K)
    features_gpu = F.normalize(
        TRAIN_FEATURES.flatten(1).to(device=HEAD_DEVICE, dtype=torch.float16), dim=1)
    folds_gpu = torch.as_tensor(FOLD_VALUES, device=HEAD_DEVICE)
    labels_gpu = torch.as_tensor(LABEL_VALUES, device=HEAD_DEVICE)
    neighbour_labels = torch.empty((len(train_df), max_k), dtype=torch.long)
    for start in tqdm(range(0, len(train_df), SIMILARITY_CHUNK_ROWS), desc='neighbours'):
        end = min(start + SIMILARITY_CHUNK_ROWS, len(train_df))
        similarity = torch.matmul(features_gpu[start:end], features_gpu.t()).float()
        same_fold = folds_gpu.unsqueeze(0) == folds_gpu[start:end].unsqueeze(1)
        similarity.masked_fill_(same_fold, float('-inf'))     # also removes self
        top_indices = torch.topk(similarity, max_k, dim=1).indices
        neighbour_labels[start:end] = labels_gpu[top_indices].cpu()
        del similarity, same_fold, top_indices
    del features_gpu
    release_cuda_cache()
    neighbour_labels_np = neighbour_labels.numpy()

    # ---- D3: k-NN probe --------------------------------------------------------
    knn_results = {}
    for k in KNN_NEIGHBOUR_COUNTS:
        votes = np.zeros((len(train_df), NUM_CLASSES), dtype=np.int64)
        for class_index in range(NUM_CLASSES):
            votes[:, class_index] = (neighbour_labels_np[:, :k] == class_index).sum(axis=1)
        knn_predictions = votes.argmax(axis=1)
        knn_results[str(k)] = {
            'macro_f1': float(f1_score(LABEL_VALUES, knn_predictions,
                                       labels=list(range(NUM_CLASSES)), average='macro')),
            'accuracy': float(accuracy_score(LABEL_VALUES, knn_predictions)),
            'total_errors': int((knn_predictions != LABEL_VALUES).sum()),
        }
    nearest_label = neighbour_labels_np[:, 0]
    error_mask = reference_predictions != LABEL_VALUES
    nn_agrees_with_truth = int((nearest_label[error_mask] == LABEL_VALUES[error_mask]).sum())
    nn_agrees_with_prediction = int(
        (nearest_label[error_mask] == reference_predictions[error_mask]).sum())
    print()
    print('D3 fold-respecting k-NN probe on the raw 8192-D features:')
    for k in KNN_NEIGHBOUR_COUNTS:
        row = knn_results[str(k)]
        print('    k={:<3} macro_f1={:.10f} errors={}'.format(k, row['macro_f1'], row['total_errors']))
    print('    champion head  macro_f1={:.10f} errors={}'.format(
        reference_oof['macro_f1'], reference_oof['total_errors']))
    print('    On the {} champion errors: nearest neighbour carries the TRUE label {} times, '
          'the PREDICTED (wrong) label {} times.'.format(
              int(error_mask.sum()), nn_agrees_with_truth, nn_agrees_with_prediction))
    print('    A high "predicted" count means those images genuinely sit in the wrong region of')
    print('    embedding space -- a representation problem no head can fix.')

    # ---- D6: neighbourhood label purity (no label is ever changed) --------------
    purity = (neighbour_labels_np[:, :PURITY_K] == LABEL_VALUES[:, None]).mean(axis=1)
    low_purity_mask = purity <= PURITY_LOW_THRESHOLD
    purity_summary = {
        'k': PURITY_K, 'low_threshold': PURITY_LOW_THRESHOLD,
        'rows_with_low_purity': int(low_purity_mask.sum()),
        'low_purity_and_champion_error': int((low_purity_mask & error_mask).sum()),
        'low_purity_but_champion_correct': int((low_purity_mask & (~error_mask)).sum()),
        'champion_errors_with_high_purity': int(((~low_purity_mask) & error_mask).sum()),
        'mean_purity_all': float(purity.mean()),
        'mean_purity_on_champion_errors': float(purity[error_mask].mean()) if error_mask.any() else None,
        'by_class': {CLASS_NAMES[c]: int((low_purity_mask & (LABEL_VALUES == c)).sum())
                     for c in range(NUM_CLASSES)},
    }
    print()
    print('D6 neighbourhood label purity (k={}), the label-noise / ambiguity floor:'.format(PURITY_K))
    print('    rows whose neighbourhood almost never shares their label (purity <= {:.2f}): {}'.format(
        PURITY_LOW_THRESHOLD, purity_summary['rows_with_low_purity']))
    print('      of those, champion already gets wrong : {}'.format(
        purity_summary['low_purity_and_champion_error']))
    print('      of those, champion still gets right   : {}'.format(
        purity_summary['low_purity_but_champion_correct']))
    print('    champion errors that sit in a CLEAN neighbourhood : {}'.format(
        purity_summary['champion_errors_with_high_purity']))
    print('    mean purity overall {:.4f}, on champion errors {:.4f}'.format(
        purity_summary['mean_purity_all'],
        purity_summary['mean_purity_on_champion_errors'] or float('nan')))
    print('    READ THIS AS: low-purity rows are where the label and the visual evidence disagree.')
    print('    They are NOT relabelled and NOT dropped -- this is a measurement, not an intervention.')

    # ---- D5: size and aspect stratification ------------------------------------
    strata = train_df[['label', 'class_name', 'is_upsampled', 'aspect_ratio']].copy()
    strata['error'] = error_mask
    strata['aspect_bucket'] = pd.cut(strata['aspect_ratio'], [0, 1.2, 1.6, 2.5, 1e9],
                                     labels=['~square', 'mild', 'strong', 'extreme'])
    by_class_size = strata.groupby(['class_name', 'is_upsampled'], observed=True)['error'].agg(
        ['size', 'sum', 'mean']).reset_index()
    by_aspect = strata.groupby(['aspect_bucket'], observed=True)['error'].agg(
        ['size', 'sum', 'mean']).reset_index()
    write_csv_atomic(by_class_size, ARTIFACT_DIR / 'error_rate_by_class_and_size.csv')
    write_csv_atomic(by_aspect, ARTIFACT_DIR / 'error_rate_by_aspect.csv')
    print()
    print('D5 error rate by class x upsampled:')
    display(by_class_size)
    print('D5 error rate by aspect-ratio bucket:')
    display(by_aspect)

    # ---- D4: PCA projection figure ---------------------------------------------
    try:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt

        projector = PCA(n_components=2, random_state=FOLD_SEED, svd_solver='randomized')
        coordinates = projector.fit_transform(
            TRAIN_FEATURES.flatten(1).to(torch.float32).numpy())
        figure, axis = plt.subplots(figsize=(9, 8))
        for class_index, colour in zip(range(NUM_CLASSES), ('tab:blue', 'tab:green', 'tab:orange')):
            mask = LABEL_VALUES == class_index
            axis.scatter(coordinates[mask, 0], coordinates[mask, 1], s=2, alpha=0.25,
                         c=colour, label=CLASS_NAMES[class_index])
        axis.scatter(coordinates[error_mask, 0], coordinates[error_mask, 1], s=14, alpha=0.9,
                     facecolors='none', edgecolors='red', linewidths=0.6,
                     label='champion errors ({})'.format(int(error_mask.sum())))
        axis.set_title('DINOv3 ViT-7B frozen features, PCA-2D (explained variance {:.3f})'.format(
            float(projector.explained_variance_ratio_.sum())))
        axis.legend(markerscale=3, loc='best')
        figure.tight_layout()
        figure.savefig(FIGURES_DIR / 'pca_projection.png', dpi=130)
        plt.close(figure)
        pca_explained = float(projector.explained_variance_ratio_.sum())
        print()
        print('D4 PCA-2D figure written to figures/pca_projection.png '
              '(explained variance {:.3f}).'.format(pca_explained))
    except Exception as figure_error:
        pca_explained = None
        print()
        print('D4 PCA figure skipped: {}: {}'.format(type(figure_error).__name__, figure_error))

    DIAGNOSTICS = {
        'attempted': True, 'succeeded': True, 'is_a_gate': False,
        'labels_modified': False, 'rows_dropped': 0,
        'd1_seed_consensus': seed_consensus,
        'd1_always_wrong': always_wrong, 'd1_flips_between_seeds': sometimes_wrong,
        'd2_error_counts_per_source': cross_experiment,
        'd2_universal_error_rows': len(universal_errors),
        'd3_knn': knn_results,
        'd3_champion_macro_f1': reference_oof['macro_f1'],
        'd3_nn_label_on_champion_errors': {'true_class': nn_agrees_with_truth,
                                           'predicted_class': nn_agrees_with_prediction,
                                           'total_errors': int(error_mask.sum())},
        'd4_pca_explained_variance': pca_explained,
        'd5_error_rate_by_class_and_size': by_class_size.to_dict('records'),
        'd5_error_rate_by_aspect': [{k: (str(v) if k == 'aspect_bucket' else v) for k, v in row.items()}
                                    for row in by_aspect.to_dict('records')],
        'd6_purity': purity_summary,
    }
    write_json_atomic(DIAGNOSTICS, ARTIFACT_DIR / 'ceiling_diagnostics.json')
    write_csv_atomic(pd.DataFrame({
        'image_id': train_df['image_id'], 'label': LABEL_VALUES,
        'champion_prediction': reference_predictions, 'ensemble_prediction': ensemble_predictions,
        'wrong_in_n_seeds': wrong_count, 'neighbour_purity': purity,
        'nearest_neighbour_label': nearest_label, 'is_upsampled': train_df['is_upsampled'],
        'aspect_ratio': train_df['aspect_ratio'],
    }), ARTIFACT_DIR / 'per_row_diagnostics.csv')
    del neighbour_labels, neighbour_labels_np
    release_cuda_cache()
except Exception as diagnostic_error:
    DIAGNOSTICS = {'attempted': True, 'succeeded': False, 'is_a_gate': False,
                   'error_type': type(diagnostic_error).__name__, 'error': str(diagnostic_error)}
    write_json_atomic(DIAGNOSTICS, ARTIFACT_DIR / 'ceiling_diagnostics.json')
    release_cuda_cache()
    print()
    print('Ceiling diagnostics FAILED: {}: {}'.format(
        type(diagnostic_error).__name__, diagnostic_error))
    print('Diagnostic-only and deliberately non-fatal. The adoption gate is unaffected.')

print()
print('Elapsed: {:.0f} min'.format(elapsed_minutes()))

In [ ]:
SELECTED_MODE = 'seed_ensemble' if ENSEMBLE_ADOPTED else 'single_seed_champion'
SELECTED_SEEDS = list(ENSEMBLE_SEEDS) if ENSEMBLE_ADOPTED else [REFERENCE_SEED]
SELECTION_REASON = (
    'the locked {}-seed ensemble cleared every gate'.format(ENSEMBLE_K) if ENSEMBLE_ADOPTED
    else 'the seed ensemble failed at least one locked gate; the single-seed V10 champion stands')

SELECTION_DECISION = {
    'schema_version': 1, 'revision_marker': REVISION_MARKER,
    'selected_mode': SELECTED_MODE, 'selected_seeds': SELECTED_SEEDS,
    'aggregation_rule': AGGREGATION_RULE if ENSEMBLE_ADOPTED else 'single model argmax',
    'selection_reason': SELECTION_REASON, 'selection_basis': 'train_only_cross_validation',
    'ensemble_adopted': ENSEMBLE_ADOPTED, 'adoption_gate_checks': ADOPTION_GATE_CHECKS,
    'locked_seeds': list(ENSEMBLE_SEEDS), 'locked_k': ENSEMBLE_K,
    'best_single_seed': BEST_SINGLE_SEED,
    'single_seed_macro_f1': {str(s): SEED_OOF[s]['macro_f1'] for s in ENSEMBLE_SEEDS},
    'single_seed_total_errors': {str(s): SEED_OOF[s]['total_errors'] for s in ENSEMBLE_SEEDS},
    'mean_single_seed_macro_f1': MEAN_SINGLE_MACRO,
    'observed_seed_std': OBSERVED_SEED_STD, 'reference_seed_std_from_v13': SEED_NOISE_STD,
    'ensemble_macro_f1': ENSEMBLE_OOF['macro_f1'],
    'ensemble_total_errors': ENSEMBLE_OOF['total_errors'],
    'ensemble_ro_errors': ENSEMBLE_OOF['ro_errors'],
    'gain_over_best_single_seed': GAIN_OVER_BEST_SEED,
    'gain_over_mean_single_seed': GAIN_OVER_MEAN_SEED,
    'development_fold_gain': DEVELOPMENT_FOLD_GAIN,
    'assessment_fold_gains': ASSESSMENT_FOLD_GAINS, 'assessment_folds_won': FOLDS_WON,
    'electronic_f1_drop': ELECTRONIC_F1_DROP,
    'leave_one_seed_out': LOSO_RESULTS,
    'ensemble_oof_metrics': ENSEMBLE_OOF,
    'per_seed_oof_metrics': {str(s): SEED_OOF[s] for s in ENSEMBLE_SEEDS},
    'control_reproduction': CONTROL_REPRODUCTION,
    'ceiling_diagnostics': DIAGNOSTICS,
    'cache_acquisition': CACHE_ACQUISITION,
    'fold_ensembling_used': False, 'model_soup_used': False,
    'cross_architecture_blending_used': False, 'threshold_tuning_used': False,
    'labels_modified': False, 'rows_dropped': 0,
    'data_contract_signature': DATA_CONTRACT_SIGNATURE,
    'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE,
    'test_images_accessed': TEST_ACCESS_STATE['test_images_accessed'],
}
SELECTION_DECISION_SIGNATURE = sha256_json(SELECTION_DECISION)
write_json_atomic({'selection_decision': SELECTION_DECISION,
                   'selection_decision_signature': SELECTION_DECISION_SIGNATURE},
                  SELECTION_DECISION_PATH)
print('selection_decision.json written BEFORE any test image is opened.')
print('Selected mode:', SELECTED_MODE, '| seeds:', SELECTED_SEEDS)
print('Signature    :', SELECTION_DECISION_SIGNATURE)


def train_full_head(seed, epochs):
    """Retrain from scratch on every official raw training row. No validation, no early stopping."""
    all_indices = np.arange(len(train_df), dtype=np.int64)
    class_counts, class_weights = class_weights_from(LABEL_VALUES)
    seed_everything(seed)
    model = ChampionHead().to(HEAD_DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=HEAD_LEARNING_RATE, weight_decay=HEAD_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

    def on_epoch_end(epoch, train_loss, learning_rate, epoch_seconds):
        print('    full_training seed={} epoch={:02d}/{:02d} loss={:.6f}'.format(
            seed, epoch, epochs, train_loss))
        return False

    run_training_epochs(model, TRAIN_FEATURES, all_indices, epochs, seed,
                        criterion, optimizer, scheduler, on_epoch_end)
    del criterion, optimizer, scheduler
    return model


print()
print('=' * 78)
print('FULL-DATA RETRAIN   [t+{:.0f} min]'.format(elapsed_minutes()))
print('=' * 78)
FINAL_HEADS, FINAL_HEAD_EPOCHS = {}, {}
for seed in SELECTED_SEEDS:
    best_epochs = [int(m['best_epoch']) for m in SEED_PER_FOLD[seed]]
    epochs = int(round(float(np.median(best_epochs))))
    if not 1 <= epochs <= MAX_EPOCHS:
        raise RuntimeError('Derived epoch count {} for seed {} is out of range.'.format(epochs, seed))
    FINAL_HEAD_EPOCHS[seed] = epochs
    print('  seed {} per-fold best epochs {} -> median {}'.format(seed, best_epochs, epochs))
    FINAL_HEADS[seed] = train_full_head(seed, epochs)

FINAL_HEAD_PATH = ARTIFACT_DIR / 'final_heads.pt'
atomic_torch_save({
    'metadata': {
        'schema_version': 1, 'revision_marker': REVISION_MARKER, 'mode': SELECTED_MODE,
        'seeds': SELECTED_SEEDS, 'epochs_per_seed': {str(s): FINAL_HEAD_EPOCHS[s] for s in SELECTED_SEEDS},
        'aggregation': AGGREGATION_RULE if ENSEMBLE_ADOPTED else 'single model argmax',
        'epoch_selection': 'median best epoch across that seed cross-validation folds',
        'trained_on': 'all {} official raw training rows'.format(len(train_df)),
        'early_stopping_used': False, 'warm_start_used': False, 'labels_modified': False,
        'head_architecture': 'LayerNorm(8192) -> Linear(512) -> GELU -> Dropout(0.4) -> Linear(3)',
        'selection_decision_signature': SELECTION_DECISION_SIGNATURE,
        'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE,
    },
    'state_dicts': {str(s): {n: t.detach().cpu().clone() for n, t in FINAL_HEADS[s].state_dict().items()}
                    for s in SELECTED_SEEDS},
}, FINAL_HEAD_PATH)
print()
print('Final head bundle written: {} ({} head{})'.format(
    FINAL_HEAD_PATH, len(SELECTED_SEEDS), '' if len(SELECTED_SEEDS) == 1 else 's'))

# ---------------------------------------------------------------------------
# TEST ISOLATION RELEASE
# ---------------------------------------------------------------------------
persisted = read_json(SELECTION_DECISION_PATH) if SELECTION_DECISION_PATH.is_file() else {}
TEST_UNLOCK_CHECKS = {
    'selection_decision_file_exists': bool(persisted),
    'selection_decision_signature_matches': (
        persisted.get('selection_decision_signature') == SELECTION_DECISION_SIGNATURE),
    'all_seed_oof_complete': all(SEED_OOF[s]['rows'] == EXPECTED_TRAIN_ROWS for s in ENSEMBLE_SEEDS),
    'final_head_bundle_exists': FINAL_HEAD_PATH.is_file(),
    'test_untouched_during_selection': (
        persisted.get('selection_decision', {}).get('test_images_accessed') is False),
}
failed_unlock = [k for k, v in TEST_UNLOCK_CHECKS.items() if not v]
if failed_unlock:
    raise RuntimeError('Test isolation cannot be released. Failed: {}.'.format(failed_unlock))
TEST_ACCESS_STATE['test_images_accessed'] = True
print()
print('Test isolation released; all train-only prerequisites passed.')


def valid_test_dir(candidate):
    try:
        entries = list(candidate.iterdir())
    except OSError:
        return False
    return len(entries) == EXPECTED_TEST_ROWS and all(
        e.is_file() and not e.is_symlink() and e.suffix.lower() in ALLOWED_IMAGE_SUFFIXES for e in entries)


def resolve_test_dir():
    candidates = [Path.cwd() / 'data' / 'test', KAGGLE_WORKING / 'data' / 'test']
    candidates.extend(discover_named_dirs(KAGGLE_INPUT, 'test'))
    matches = {c.resolve() for c in candidates if c.is_dir() and valid_test_dir(c)}
    if len(matches) != 1:
        rendered = '\n'.join('  - {}'.format(p) for p in sorted(matches, key=str)) or '  (none)'
        raise RuntimeError('Expected exactly one official test directory with {} images; found {}:'.format(
            EXPECTED_TEST_ROWS, len(matches)) + '\n' + rendered)
    return matches.pop()


def resolve_submission_template():
    matches = {}
    for root in (Path.cwd() / 'data', KAGGLE_WORKING / 'data', KAGGLE_INPUT):
        if not root.is_dir():
            continue
        for candidate in root.rglob('*.csv'):
            try:
                frame = pd.read_csv(candidate)
            except Exception:
                continue
            if list(frame.columns) == SUBMISSION_COLUMNS and len(frame) == EXPECTED_TEST_ROWS:
                matches[str(candidate.resolve())] = frame
    if len(matches) != 1:
        raise RuntimeError('Expected exactly one official submission template; found {}.'.format(len(matches)))
    path = next(iter(matches))
    return Path(path), matches[path]


TEST_DIR = resolve_test_dir()
TEMPLATE_PATH, submission_template = resolve_submission_template()
stem_to_path = {}
for path in sorted(TEST_DIR.iterdir(), key=lambda p: p.name):
    if path.stem in stem_to_path:
        raise RuntimeError('Duplicate test image stem: {}'.format(path.stem))
    stem_to_path[path.stem] = path
template_ids = submission_template['id'].tolist()
if len(set(template_ids)) != EXPECTED_TEST_ROWS:
    raise RuntimeError('The official template contains duplicate ids.')
ordered_paths = []
for raw_id in template_ids:
    key = Path(str(raw_id)).stem
    if key not in stem_to_path:
        raise RuntimeError('Template id {!r} has no matching image in {}.'.format(raw_id, TEST_DIR))
    ordered_paths.append(str(stem_to_path[key].resolve()))
test_df = pd.DataFrame({'image_id': [str(i) for i in template_ids], 'path': ordered_paths})
TEST_ORDER_SHA256 = sha256_json(test_df['image_id'].tolist())

# Prefer a test cache that sits in the SAME bundle as the accepted train cache: that bundle
# already proved itself by reproducing the champion, so its sibling inherits the trust.
TEST_FEATURES = None
if CACHE_ACQUISITION['train']['source'] == 'mounted_cache':
    sibling = Path(CACHE_ACQUISITION['train']['directory']).parent / 'test'
    if sibling.is_dir():
        candidate_features = load_cache_dir(sibling, 'test', test_df)
        if candidate_features is not None:
            TEST_FEATURES = candidate_features
            CACHE_ACQUISITION['test'] = {'source': 'mounted_cache_sibling', 'directory': str(sibling)}
            print('Test features reused from the same verified bundle: {}'.format(sibling))
if TEST_FEATURES is None:
    print('Extracting test features from the frozen backbone ({} images)...'.format(EXPECTED_TEST_ROWS))
    TEST_FEATURES = extract_split(test_df, 'test')
    CACHE_ACQUISITION['test'] = {'source': 'fresh_extraction', 'directory': str(CACHE_DIR / 'test')}
if tuple(TEST_FEATURES.shape) != (EXPECTED_TEST_ROWS,) + FIRST_ORDER_TAIL:
    raise RuntimeError('Test feature tensor has shape {}.'.format(tuple(TEST_FEATURES.shape)))
write_json_atomic(CACHE_ACQUISITION, ARTIFACT_DIR / 'cache_acquisition.json')
release_cuda_cache()

test_indices = np.arange(len(test_df), dtype=np.int64)
TEST_INFERENCE_STARTED_AT = time.perf_counter()
per_seed_test = {}
for seed in SELECTED_SEEDS:
    per_seed_test[seed] = predict_probabilities(FINAL_HEADS[seed], TEST_FEATURES, test_indices)
    write_npy_atomic(per_seed_test[seed].astype(np.float32),
                     ARTIFACT_DIR / 'test_probabilities_seed{}.npy'.format(seed))
stacked_test = np.stack([per_seed_test[s] for s in SELECTED_SEEDS], axis=0)
test_probabilities = stacked_test.mean(axis=0)
test_probabilities = test_probabilities / test_probabilities.sum(axis=1, keepdims=True)
TEST_INFERENCE_SECONDS = time.perf_counter() - TEST_INFERENCE_STARTED_AT
if test_probabilities.shape != (EXPECTED_TEST_ROWS, NUM_CLASSES) or not np.isfinite(test_probabilities).all():
    raise RuntimeError('Test probability matrix is incomplete or non-finite.')
write_npy_atomic(test_probabilities.astype(np.float32), ARTIFACT_DIR / 'test_probabilities.npy')

test_predictions = test_probabilities.argmax(axis=1).astype(np.int64)
submission = pd.DataFrame({'id': submission_template['id'].to_numpy(), 'predicted': test_predictions})
SUBMISSION_VALIDATION = {
    'row_count_is_expected': bool(len(submission) == EXPECTED_TEST_ROWS),
    'columns_are_exact': bool(list(submission.columns) == SUBMISSION_COLUMNS),
    'no_missing_values': bool(not submission.isna().any().any()),
    'ids_are_unique': bool(submission['id'].is_unique),
    'predictions_in_class_codes': bool(submission['predicted'].isin([0, 1, 2]).all()),
    'row_order_matches_template': bool(submission['id'].tolist() == submission_template['id'].tolist()),
}
failed_validation = [k for k, v in SUBMISSION_VALIDATION.items() if not v]
if failed_validation:
    raise RuntimeError('Submission validation failed: {}.'.format(failed_validation))
SUBMISSION_PATH = ARTIFACT_DIR / 'submission_{}.csv'.format(sanitize_for_filename(TEAM_NAME))
write_csv_atomic(submission, SUBMISSION_PATH)
PREDICTED_DISTRIBUTION = {str(c): int((test_predictions == c).sum()) for c in range(NUM_CLASSES)}

print()
print('Test inference: one frozen-backbone forward pass per image; the 4.2M head evaluated {} time{}.'.format(
    len(SELECTED_SEEDS), '' if len(SELECTED_SEEDS) == 1 else 's'))
print('  no TTA, no hflip, no multi-crop, no fold averaging, no cross-architecture blending,')
print('  no manual correction, no threshold tuning.')
print('Predicted class distribution :', PREDICTED_DISTRIBUTION)
print('V10 test distribution        :', V10_REFERENCE['test_predicted_class_distribution'])
print('  (diagnostic only; it never influenced training, selection or the predictions.)')
print('Submission written           :', SUBMISSION_PATH)
print()
print('!! DO NOT UPLOAD YET. Per the experiment-cycle model, the final candidate is chosen near')
print('!! the deadline from all gate-passing candidates. Only one submission remains.')

In [ ]:
# ---------------------------------------------------------------------------
# REPRODUCTION CHECK from the persisted bundle. V10 precedent: 2.98e-08.
# ---------------------------------------------------------------------------
reloaded = torch.load(FINAL_HEAD_PATH, map_location='cpu', weights_only=False)
reproduced_per_seed = []
for seed in SELECTED_SEEDS:
    head = ChampionHead().to(HEAD_DEVICE)
    head.load_state_dict(reloaded['state_dicts'][str(seed)], strict=True)
    reproduced_per_seed.append(predict_probabilities(head, TEST_FEATURES, test_indices))
    del head
reproduced = np.stack(reproduced_per_seed, axis=0).mean(axis=0)
reproduced = reproduced / reproduced.sum(axis=1, keepdims=True)
MAX_ABS_PROBABILITY_DIFFERENCE = float(np.abs(reproduced - test_probabilities).max())
PREDICTIONS_IDENTICAL = bool(np.array_equal(reproduced.argmax(axis=1), test_predictions))
CHECKPOINT_REPRODUCTION = {
    'checkpoint': str(FINAL_HEAD_PATH), 'seeds': SELECTED_SEEDS,
    'max_abs_probability_difference': MAX_ABS_PROBABILITY_DIFFERENCE,
    'tolerance': RELOAD_TOLERANCE,
    'within_tolerance': bool(MAX_ABS_PROBABILITY_DIFFERENCE <= RELOAD_TOLERANCE),
    'identical_argmax': PREDICTIONS_IDENTICAL,
    'v10_precedent_max_abs_probability_difference': 2.98e-08,
}
write_json_atomic(CHECKPOINT_REPRODUCTION, ARTIFACT_DIR / 'checkpoint_reproduction_check.json')
if not (CHECKPOINT_REPRODUCTION['within_tolerance'] and PREDICTIONS_IDENTICAL):
    raise RuntimeError('Checkpoint reproduction failed: max abs diff {:.3e}, identical argmax {}.'.format(
        MAX_ABS_PROBABILITY_DIFFERENCE, PREDICTIONS_IDENTICAL))
del reloaded, reproduced_per_seed
release_cuda_cache()

# ---------------------------------------------------------------------------
# ERROR ANALYSIS on the selected OOF (train only, never test)
# ---------------------------------------------------------------------------
selected_oof_probabilities = ENSEMBLE_OOF_PROBABILITIES if ENSEMBLE_ADOPTED \
    else SEED_PROBABILITIES[REFERENCE_SEED]
selected_oof = ENSEMBLE_OOF if ENSEMBLE_ADOPTED else SEED_OOF[REFERENCE_SEED]
selected_predictions = selected_oof_probabilities.argmax(axis=1)
error_rows = np.flatnonzero(selected_predictions != LABEL_VALUES)
top_two = np.sort(selected_oof_probabilities, axis=1)[:, -2:]
error_frame = pd.DataFrame({
    'image_id': train_df['image_id'].to_numpy()[error_rows],
    'label': LABEL_VALUES[error_rows], 'prediction': selected_predictions[error_rows],
    'confidence': top_two[error_rows, 1], 'margin': top_two[error_rows, 1] - top_two[error_rows, 0],
    'fold': FOLD_VALUES[error_rows], 'is_upsampled': train_df['is_upsampled'].to_numpy()[error_rows],
}).sort_values('confidence', ascending=False)
write_csv_atomic(error_frame, ARTIFACT_DIR / 'oof_errors_selected.csv')

reference_wrong = SEED_PROBABILITIES[REFERENCE_SEED].argmax(axis=1) != LABEL_VALUES
ensemble_wrong = ENSEMBLE_OOF_PROBABILITIES.argmax(axis=1) != LABEL_VALUES
ERROR_ANALYSIS = {
    'selected_mode': SELECTED_MODE,
    'oof_total_errors': selected_oof['total_errors'], 'oof_ro_errors': selected_oof['ro_errors'],
    'oof_electronic_errors': selected_oof['electronic_errors'],
    'oof_high_confidence_errors': selected_oof['high_confidence_errors'],
    'rows_fixed_by_ensemble': int((reference_wrong & (~ensemble_wrong)).sum()),
    'rows_broken_by_ensemble': int(((~reference_wrong) & ensemble_wrong).sum()),
    'net_rows_changed_by_ensemble': int((reference_wrong & (~ensemble_wrong)).sum()
                                        - ((~reference_wrong) & ensemble_wrong).sum()),
    'historical_v10_oof_total_errors': V10_REFERENCE['oof_total_errors'],
    'label_ceiling_note': (
        'Validation always uses the original official labels, so a true-accuracy improvement that crosses '
        'a mislabelled row LOWERS measured OOF. Read D6 in ceiling_diagnostics.json alongside this.'),
    'labels_modified': False, 'rows_dropped': 0,
    'test_images_inspected_for_model_decisions': False,
}
write_json_atomic(ERROR_ANALYSIS, ARTIFACT_DIR / 'error_analysis.json')

write_json_atomic({
    'schema_version': 1, 'revision_marker': REVISION_MARKER,
    'selected_mode': SELECTED_MODE, 'selected_seeds': SELECTED_SEEDS,
    'aggregation': AGGREGATION_RULE if ENSEMBLE_ADOPTED else 'single model argmax',
    'selection_reason': SELECTION_REASON,
    'selection_decision_signature': SELECTION_DECISION_SIGNATURE,
    'decision_rule': 'argmax over the aggregated probability matrix',
    'backbone_forward_passes_per_image': 1, 'head_evaluations_per_image': len(SELECTED_SEEDS),
    'inference_seconds': TEST_INFERENCE_SECONDS,
    'final_head_bundle': str(FINAL_HEAD_PATH), 'epochs_per_seed': FINAL_HEAD_EPOCHS,
    'fold_ensembling': False, 'cross_architecture_blending': False, 'model_soup': False,
    'test_time_augmentation': False, 'multi_crop_inference': False,
    'horizontal_flip_inference': False, 'manual_prediction_correction': False,
    'leaderboard_feedback_used': False, 'previous_submissions_loaded': False,
    'predicted_class_distribution': PREDICTED_DISTRIBUTION,
    'mean_max_probability': float(test_probabilities.max(axis=1).mean()),
    'submission_path': str(SUBMISSION_PATH), 'submission_validation': SUBMISSION_VALIDATION,
    'template_path': str(TEMPLATE_PATH), 'test_dir': str(TEST_DIR),
    'test_order_sha256': TEST_ORDER_SHA256,
    'checkpoint_reproduction': CHECKPOINT_REPRODUCTION,
    'cache_acquisition': CACHE_ACQUISITION,
    'submitted_to_organisers': False,
    'submission_policy_note': 'one submission remains; the final candidate is chosen by the user near the deadline',
}, ARTIFACT_DIR / 'test_inference_manifest.json')

write_json_atomic({
    'schema_version': 1, 'revision_marker': REVISION_MARKER, 'experiment_name': EXPERIMENT_NAME,
    'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE,
    'data_contract_signature': DATA_CONTRACT_SIGNATURE,
    'selection_decision_signature': SELECTION_DECISION_SIGNATURE,
    'fold_assignment_sha256': FOLD_ASSIGNMENT_SHA256, 'labels_sha256': LABELS_SHA256,
    'ordered_image_ids_sha256': ORDERED_IMAGE_IDS_SHA256, 'test_order_sha256': TEST_ORDER_SHA256,
    'environment': ENVIRONMENT_INFO, 'gpus': GPU_RUNTIME_INFO,
    'total_runtime_minutes': elapsed_minutes(),
}, ARTIFACT_DIR / 'reproducibility_manifest.json')

# ---------------------------------------------------------------------------
# THE ONE BLOCK TO COPY BACK
# ---------------------------------------------------------------------------
print()
print('#' * 78)
print('# V15 RESULT BLOCK  --  copy this whole block back')
print('#' * 78)
print('experiment                     : {}'.format(EXPERIMENT_NAME))
print('feature source                 : train={} test={}'.format(
    CACHE_ACQUISITION['train']['source'], (CACHE_ACQUISITION['test'] or {}).get('source')))
print('pipeline integrity (seed {})    : {}   delta {:+.10f}  (exact: {})'.format(
    REFERENCE_SEED, 'PASS' if PIPELINE_INTEGRITY_OK else 'FAIL',
    CONTROL_REPRODUCTION['macro_f1_delta'], CONTROL_REPRODUCTION['exactly_reproduced']))
print('---- per-seed OOF (locked seeds {}) ----'.format(list(ENSEMBLE_SEEDS)))
for seed in ENSEMBLE_SEEDS:
    oof = SEED_OOF[seed]
    print('  seed {:<5} macro_f1={:.10f} err={} R/O={} elecF1={:.10f} capped={}/5'.format(
        seed, oof['macro_f1'], oof['total_errors'], oof['ro_errors'],
        oof['f1_electronic'], oof['folds_hitting_epoch_cap']))
print('  best single seed             : {} at {:.10f}'.format(BEST_SINGLE_SEED,
                                                              BEST_SINGLE_OOF['macro_f1']))
print('  mean single seed             : {:.10f}  (observed std {:.8f})'.format(
    MEAN_SINGLE_MACRO, OBSERVED_SEED_STD))
print('---- ENSEMBLE k={} ----'.format(ENSEMBLE_K))
print('  OOF macro F1                 : {:.10f}'.format(ENSEMBLE_OOF['macro_f1']))
print('  OOF total errors             : {}'.format(ENSEMBLE_OOF['total_errors']))
print('  OOF R/O errors               : {}  (R->O {}, O->R {})'.format(
    ENSEMBLE_OOF['ro_errors'], ENSEMBLE_OOF['recyclable_to_organic'],
    ENSEMBLE_OOF['organic_to_recyclable']))
print('  OOF Electronic F1            : {:.10f}  (errors {})'.format(
    ENSEMBLE_OOF['f1_electronic'], ENSEMBLE_OOF['electronic_errors']))
print('  log loss                     : {:.10f}'.format(ENSEMBLE_OOF['log_loss']))
print('  gain over best single seed   : {:+.10f}   (gate {:+.6f})'.format(
    GAIN_OVER_BEST_SEED, ADOPTION_MIN_GAIN_OVER_BEST_SEED))
print('  gain over mean single seed   : {:+.10f}'.format(GAIN_OVER_MEAN_SEED))
print('  fold 0 gain                  : {:+.10f}'.format(DEVELOPMENT_FOLD_GAIN))
print('  assessment folds won         : {} of 4'.format(FOLDS_WON))
print('  rows fixed / broken          : {} / {} (net {})'.format(
    ERROR_ANALYSIS['rows_fixed_by_ensemble'], ERROR_ANALYSIS['rows_broken_by_ensemble'],
    ERROR_ANALYSIS['net_rows_changed_by_ensemble']))
print('  leave-one-seed-out all win   : {}'.format(LOSO_ALL_WIN))
print('---- CEILING DIAGNOSTICS ----')
if DIAGNOSTICS.get('succeeded'):
    print('  wrong in ALL {} seeds         : {}   (ensembling can never fix these)'.format(
        ENSEMBLE_K, DIAGNOSTICS['d1_always_wrong']))
    print('  flips between seeds          : {}   (the entire ensembling headroom)'.format(
        DIAGNOSTICS['d1_flips_between_seeds']))
    print('  wrong in EVERY model found   : {} across {} prediction sets'.format(
        DIAGNOSTICS['d2_universal_error_rows'], len(DIAGNOSTICS['d2_error_counts_per_source']) + 1))
    for k, row in sorted(DIAGNOSTICS['d3_knn'].items(), key=lambda kv: int(kv[0])):
        print('  k-NN k={:<3}                   : macro_f1 {:.10f}, errors {}'.format(
            k, row['macro_f1'], row['total_errors']))
    nn_row = DIAGNOSTICS['d3_nn_label_on_champion_errors']
    print('  on champion errors, NN label : true {} / predicted {} of {}'.format(
        nn_row['true_class'], nn_row['predicted_class'], nn_row['total_errors']))
    purity_row = DIAGNOSTICS['d6_purity']
    print('  low-purity rows (<= {:.2f})     : {}  [champion wrong on {}, right on {}]'.format(
        purity_row['low_threshold'], purity_row['rows_with_low_purity'],
        purity_row['low_purity_and_champion_error'], purity_row['low_purity_but_champion_correct']))
    print('  champion errors in CLEAN nbhd: {}'.format(purity_row['champion_errors_with_high_purity']))
else:
    print('  FAILED (non-fatal): {}'.format(DIAGNOSTICS.get('error', 'unknown')))
print('---- gate ----')
for name, passed in ADOPTION_GATE_CHECKS.items():
    print('  [{}] {}'.format('PASS' if passed else 'FAIL', name))
print('---- decision ----')
print('  ensemble adopted             : {}'.format(ENSEMBLE_ADOPTED))
print('  selected mode                : {} (seeds {})'.format(SELECTED_MODE, SELECTED_SEEDS))
print('  epochs per seed              : {}'.format(FINAL_HEAD_EPOCHS))
print('  test class distribution      : {}'.format(PREDICTED_DISTRIBUTION))
print('  V10 test class distribution  : {}'.format(V10_REFERENCE['test_predicted_class_distribution']))
print('  reproduction max abs diff    : {:.3e}  (identical argmax: {})'.format(
    MAX_ABS_PROBABILITY_DIFFERENCE, PREDICTIONS_IDENTICAL))
print('  submission (NOT uploaded)    : {}'.format(SUBMISSION_PATH))
print('  total runtime                : {:.0f} min'.format(elapsed_minutes()))
print('#' * 78)

## 8. Cara membaca hasil

Salin **V15 RESULT BLOCK** apa adanya. Keputusan adopsi sudah dihitung otomatis; yang perlu dibaca manusia adalah bagian diagnostik.

### 8.1 Urutan pemeriksaan

**Langkah 1 — sumber fitur & integritas.** `feature source` memberi tahu apakah cache pinjaman diterima. `pipeline integrity` harus `PASS`; V13 dan V14 dua-duanya mencapai delta persis 0.0. Kalau cache pinjaman diterima tapi deltanya bukan 0.0 (masih dalam toleransi), catat — artinya fiturnya mirip tapi tidak identik.

**Langkah 2 — gerbang ensemble.** Sembilan baris `PASS`/`FAIL`. Adopsi hanya kalau semuanya `PASS`.

**Langkah 3 — apakah gain-nya masuk akal.** Bandingkan tiga angka:

- `gain over best single seed` — yang dipakai gerbang. Ambang +0,000426 memang berat: single-seed terbaik dari 5 undian sudah bias ke atas.
- `gain over mean single seed` — hampir pasti lebih besar, dan inilah yang lebih mendekati manfaat sebenarnya dibanding "resep V10 dengan seed acak".
- `leave-one-seed-out all win` — kalau `False`, kemenangannya bergantung pada satu seed dan tidak boleh dipercaya.

### 8.2 Diagnostik — di sinilah strategi bisa berubah

Ini bagian yang sebenarnya dicari, bukan angka ensemble-nya.

| Yang dilihat | Kalau angkanya begini | Artinya |
|---|---|---|
| `wrong in ALL K seeds` mendekati total error | ensembling hampir tidak punya ruang | seed variance bukan penyebab utama error; jangan berharap banyak dari K yang lebih besar |
| `flips between seeds` besar (mis. > 60) | ada ruang nyata | seed ensembling masuk akal, dan menaikkan K bisa membantu |
| `wrong in EVERY model found` mendekati 287 | inti yang kebal head | **berhenti mencoba head baru.** Yang tersisa hanya representasi (resolusi/geometri) atau memang plafon data |
| `k-NN k=20` mendekati macro F1 head | head sudah hampir tidak menambah apa-apa di atas geometri embedding | konfirmasi kuat bahwa readout sudah jenuh |
| `k-NN` jauh di bawah head | head masih mengekstrak sesuatu yang non-trivial | masih ada ruang, tapi V10-rich/V11/V14 menunjukkan ruangnya sempit |
| pada error champion, `NN label` lebih sering **predicted** daripada **true** | citra itu memang duduk di wilayah kelas yang salah di ruang embedding | masalah representasi, bukan masalah head |
| `low-purity rows` sebanding dengan 287 | label dan bukti visual sering bertentangan | **plafon ada di data.** Target 0,997 kemungkinan besar tidak realistis, dan itu temuan penting, bukan kegagalan |
| `champion errors in CLEAN neighbourhood` besar | banyak error terjadi di tetangga yang labelnya konsisten | itu error model yang sebenarnya bisa diperbaiki — di situ ruang perbaikan yang nyata |

Perlu ditegaskan: **tidak satu pun angka ini mengubah label.** Tidak ada relabel, tidak ada drop, tidak ada koreksi. Ini pengukuran, bukan intervensi — sesuai kebijakan baru (CONTEXT §16.5).

### 8.3 Empat kemungkinan hasil

| Hasil | Tindakan |
|---|---|
| Gerbang lolos semua, LOSO semua menang | Kandidat kuat. Catat di ledger sebagai varian `seed_ensemble_k5`. **Jangan diunggah dulu** — kandidat final dipilih di dekat deadline. |
| Gain positif tapi < +0,000426 | Tolak sesuai gerbang, tapi **catat angkanya**. Kalau gain over *mean* seed jelas positif, seed ensembling tetap menurunkan risiko walau tidak lolos ambang ketat; itu pertimbangan sah saat memilih kandidat final. |
| Gain ≈ 0 atau negatif | Seed ensembling tidak menolong di sini. Bersama D1 (`wrong in ALL K seeds`), itu berarti error-nya sistematis, bukan stokastik. |
| `pipeline integrity FAIL` | Berhenti menafsirkan. Kemungkinan besar cache pinjaman yang diterima ternyata bukan fitur yang sama; jalankan ulang tanpa melampirkan cache itu. |

### 8.4 Yang sengaja tidak dikerjakan, dan alasannya

- **Head/readout baru.** Empat kegagalan berturut-turut (V10-rich, V11, V13, V14), dan V14 membuktikan lewat kontrol capacity-matched bahwa strukturnya sendiri yang merusak. Varian kelima akan jadi tebakan.
- **Relabel / drop / koreksi label apa pun.** Dicoret permanen oleh kebijakan user.
- **LoRA pada ViT-7B.** Throughput terukur V13 5,33 citra/detik forward-only; training ≈ 4× → 1,33 citra/detik → **± 4,4 jam per epoch per fold**. Gerbang menuntut menang ≥ 3 dari 4 fold. Tidak mungkin dalam sisa waktu (CONTEXT §16.6).
- **V12 ViT-L.** Bukan ViT-7B, jadi melanggar backbone terkunci, dan submission-nya dibuat dari rata-rata 5 fold (§14.1).
- **Grouped validation sebagai eksperimen.** Sudah dijawab V14: biaya leakage hanya −0,000186 (§16.2).
- **V12B resolusi/geometri lebih tinggi.** Masih hidup sebagai hipotesis, tapi memakan ± 2,5–3 jam ekstraksi. Jalankan **hanya** kalau diagnostik menunjuk ke sana — khususnya kalau `NN label` pada error lebih sering kelas *predicted* (masalah representasi) dan error terkonsentrasi di baris ter-*upsample* dengan aspect ekstrem. Kalau justru `low-purity rows` yang besar, plafonnya data dan resolusi tidak akan menolong.

### 8.5 Artefak

```
/kaggle/working/v15_dinov3_vit7b_seed_ensemble_and_ceiling/
├── submission_<TEAM_NAME>.csv          <- fallback valid, JANGAN diunggah dulu
├── final_heads.pt                      <- bundle 1 atau 5 head, satu artefak
├── selection_decision.json             <- ditulis SEBELUM test dibuka
├── control_reproduction.json           <- gerbang integritas
├── locked_contract.json                <- K, seed, agregasi, gerbang: terkunci
├── cache_acquisition.json              <- cache mana yang diterima dan kenapa
├── ceiling_diagnostics.json            <- D1-D6, jawaban untuk "apakah 0,997 realistis"
├── per_row_diagnostics.csv             <- per baris: wrong_in_n_seeds, purity, NN label
├── error_rate_by_class_and_size.csv, error_rate_by_aspect.csv
├── seed_and_ensemble_table.csv
├── oof_probabilities_seed*.npy, oof_probabilities_ensemble.npy, oof_metrics_seed*.json
├── oof_errors_selected.csv, error_analysis.json
├── test_probabilities*.npy, test_inference_manifest.json
├── checkpoint_reproduction_check.json, reproducibility_manifest.json
└── figures/pca_projection.png
```

Simpan seluruh folder sebagai Kaggle Dataset. `per_row_diagnostics.csv` khususnya berguna untuk eksperimen berikutnya, apa pun bentuknya.